In [ ]:
# 0. Setup & shared helpers
import io, os, re, time, zipfile, json
from collections import defaultdict
import numpy as np
import pandas as pd
import requests
import duckdb
import country_converter as coco

UA = {"User-Agent": "Mozilla/5.0"}
PROJ_FROM = 2026                                       # years >= this are forward-looking where a source mixes actuals + projections
STD_COLS = ["country_code", "indicator_code", "year", "value", "source"]
FETCH_LOG = defaultdict(lambda: defaultdict(int))      # FETCH_LOG[source][status] -> count


def log_fetch(source, status):
    FETCH_LOG[source][str(status)] += 1


def print_fetch_log():
    for src, d in FETCH_LOG.items():
        print(f"  {src:<12}", dict(d))


def slug(s):
    return re.sub(r"[^A-Z0-9]+", "_", str(s).upper()).strip("_")


def tidy(df, source=None):
    """Force the master schema. Keeps `is_projection` when present."""
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=STD_COLS)
    out = df.copy()
    if source is not None:
        out["source"] = source
    out["year"] = pd.to_numeric(out["year"], errors="coerce")
    out["value"] = pd.to_numeric(out["value"], errors="coerce")
    out = out.dropna(subset=["country_code", "indicator_code", "year", "value"])
    out["year"] = out["year"].astype(int)
    cols = STD_COLS + (["is_projection"] if "is_projection" in out.columns else [])
    return out[cols]


def http_get(url, params=None, headers=None, timeout=20, tries=3, backoff=1.5):
    """GET with retry on timeouts / 429 / 5xx. Returns the last response, raises if none."""
    h = {**UA, **(headers or {})}
    last = None
    for k in range(tries):
        try:
            r = requests.get(url, params=params, headers=h, timeout=timeout)
            if r.status_code in (429, 500, 502, 503, 504):
                last = f"HTTP {r.status_code}"
                time.sleep(backoff ** (k + 1))
                continue
            return r
        except requests.RequestException as e:
            last = e
            time.sleep(backoff ** (k + 1))
    raise RuntimeError(f"GET {url} failed after {tries} tries: {last}")


def to_iso3(values, src=None):
    """Map country names (src=None -> auto-detect) or ISO2 codes (src='ISO2') to ISO3. Unknown -> None. BIS/ECB 'XM' -> 'EMU'."""
    vals = pd.Series(list(values), dtype="object")
    uniq = [v for v in vals.dropna().unique()]
    mp = {}
    if uniq:
        kw = {"src": src} if src else {}
        out = coco.convert(names=uniq, to="ISO3", not_found=None, **kw)
        out = [out] if isinstance(out, str) else out
        mp = dict(zip(uniq, out))
    mp["XM"] = "EMU"
    return vals.map(mp)


con = duckdb.connect("data/etf.duckdb", read_only=True)

try:
    df_obs = con.execute("SELECT * FROM silver.observations WHERE family = 'country'").df()
    cc = coco.CountryConverter()
    target_codes = df_obs[['metric', 'code']].drop_duplicates()
    target_codes["iso3"] = cc.pandas_convert(series=target_codes["code"], to="ISO3")
    target_countries = sorted([c for c in target_codes["iso3"].dropna().unique() if c != "not found"])
except Exception as e:
    print(f"Warning: Could not read silver.observations ({e}). Falling back to global sovereign universe.")
    cc = coco.CountryConverter()
    target_countries = sorted(cc.data['ISO3'].dropna().unique().tolist())

print(f"Initialized {len(target_countries)} target countries for ingestion loops.")

con.execute("CHECKPOINT;")
con.execute("VACUUM;")
con.close()

In [ ]:
# 1a. World Bank WDI (db 2)
# Codes that no longer exist in WDI (WGI, Doing Business, HCI, UHC, EN.ATM.CO2E.*, EG.ELC.PROD.KH, EN.CLC.*, VC.IDP.TOCV,
# SL.UEM.LTRM.ZS, IP.TMK.TOTL/NRES/RESD) are not requested here; they are covered in 1b, EIA/OWID, UNHCR and ILO below.
import wbgapi as wb

wb.get_options['timeout'] = 25
wb.get_options['headers'] = {'User-Agent': 'Mozilla/5.0'}

wb_indicators = [
    # --- health & well-being ---
    "SP.DYN.LE00.IN", "SH.DYN.MORT", "SP.DYN.IMRT.IN", "EN.ATM.PM25.MC.M3", "SH.H2O.SMDW.ZS", "SH.H2O.BASW.ZS",
    "SH.STA.SMSS.ZS", "SH.STA.BASS.ZS", "SN.ITK.DEFC.ZS", "SN.ITK.MSFI.ZS", "SN.ITK.SVFI.ZS", "VC.IHR.PSRC.P5",
    "SH.XPD.CHEX.GD.ZS", "SH.MED.PHYS.ZS", "SH.DYN.NCOM.ZS", "SH.STA.SUIC.P5",
    # --- population & workforce ---
    "SP.POP.TOTL", "SP.POP.GROW", "SP.POP.1564.TO", "SP.POP.1564.TO.ZS", "SP.POP.DPND", "SP.POP.DPND.OL",
    "SP.POP.DPND.YG", "SP.POP.65UP.TO.ZS", "SP.POP.0014.TO.ZS", "SL.TLF.TOTL.IN",
    # --- economic output & structure ---
    "NY.GDP.MKTP.PP.KD", "NY.GDP.MKTP.CD", "NY.GDP.MKTP.KD", "NY.GDP.PCAP.PP.KD", "NY.GDP.PCAP.CD", "NY.GDP.PCAP.KD",
    "NY.GDP.MKTP.KD.ZG", "NY.GDP.DEFL.KD.ZG", "SL.GDP.PCAP.EM.KD", "NV.AGR.TOTL.ZS", "NV.IND.TOTL.ZS",
    "NV.IND.MANF.ZS", "NV.IND.MANF.CD", "NV.SRV.TOTL.ZS",
    # --- trade ---
    "BX.GSR.GNFS.CD", "BM.GSR.GNFS.CD", "NE.EXP.GNFS.CD", "NE.IMP.GNFS.CD", "TX.VAL.MRCH.CD.WT", "TM.VAL.MRCH.CD.WT",
    "TX.MNF.TECH.ZS.UN", "TX.VAL.TECH.MF.ZS", "TX.VAL.TECH.CD", "NE.TRD.GNFS.ZS", "TG.VAL.TOTL.GD.ZS",
    "TM.TAX.MRCH.WM.AR.ZS", "BX.KLT.DINV.CD.WD", "BM.KLT.DINV.CD.WD",
    # --- innovation & entrepreneurship ---
    "IP.PAT.RESD", "IP.PAT.NRES", "IP.TMK.RSCT", "IP.JRN.ARTC.SC", "BX.GSR.ROYL.CD", "BM.GSR.ROYL.CD",
    "GB.XPD.RSDV.GD.ZS", "SP.POP.SCIE.RD.P6", "SP.POP.TECH.RD.P6", "NV.MNF.TECH.ZS.UN", "IC.BUS.NREG", "IC.BUS.NDNS.ZS",
    "IT.NET.USER.ZS", "IT.NET.SECR.P6", "IT.NET.BBND", "IT.NET.BBND.P2", "IT.CEL.SETS.P2", "BX.GSR.CCIS.CD",
    # --- military ---
    "MS.MIL.XPND.CD", "MS.MIL.XPND.GD.ZS", "MS.MIL.XPND.ZS", "MS.MIL.TOTL.P1", "MS.MIL.MPRT.KD", "MS.MIL.XPRT.KD",
    # --- financial markets & monetary ---
    "CM.MKT.LCAP.CD", "CM.MKT.LCAP.GD.ZS", "CM.MKT.TRAD.CD", "CM.MKT.LDOM.NO", "CM.MKT.TRNR", "FS.AST.PRVT.GD.ZS",
    "FB.BNK.CAPA.ZS", "FB.AST.NPER.ZS", "BX.PEF.TOTL.CD.WD", "FB.CBK.BRCH.P5", "FD.RES.LIQU.AS.ZS",
    "FM.LBL.BMNY.GD.ZS", "FM.LBL.BMNY.ZG", "FR.INR.RINR", "FR.INR.LEND",
    # --- education ---
    "SE.TER.ENRR", "SE.SEC.ENRR", "SE.PRM.ENRR", "SE.SEC.ENRL", "SE.PRM.ENRL", "SE.XPD.TOTL.GD.ZS", "SE.XPD.TOTL.GB.ZS",
    "SE.COM.DURS", "SE.ADT.LITR.ZS", "SE.TER.CUAT.BA.ZS", "SE.SEC.CUAT.UP.ZS", "SE.PRM.CUAT.ZS",
    # --- infrastructure & investment ---
    "NE.GDI.FTOT.CD", "NE.GDI.FTOT.ZS", "NE.GDI.FTOT.KD", "NE.GDI.TOTL.CD", "NE.GDI.TOTL.ZS", "NY.GNS.ICTR.ZS",
    "NY.GNS.ICTR.CD", "NY.GDS.TOTL.ZS", "EG.ELC.ACCS.ZS", "EG.USE.ELEC.KH.PC", "EG.FEC.RNEW.ZS", "EG.ELC.RNEW.ZS",
    "IS.AIR.PSGR", "IS.AIR.GOOD.MT.K1", "IS.RRS.GOOD.MT.K6", "IS.RRS.TOTL.KM", "IS.RRS.PASG.KM", "IS.SHP.GOOD.TU",
    "IS.SHP.GCNW.XQ", "LP.LPI.OVRL.XQ",
    # --- geology, resources, emissions ---
    "AG.LND.TOTL.K2", "ER.H2O.INTR.K3", "ER.H2O.INTR.PC", "TX.VAL.MMTL.ZS.UN", "TM.VAL.MMTL.ZS.UN",
    "AG.PRD.CREL.MT", "AG.PRD.CROP.XD", "AG.PRD.FOOD.XD", "AG.PRD.LVSK.XD", "AG.YLD.CREL.KG", "AG.LND.AGRI.ZS", "AG.LND.FRST.K2",
    "NY.GDP.TOTL.RT.ZS", "NY.GDP.PETR.RT.ZS", "NY.GDP.NGAS.RT.ZS", "NY.GDP.COAL.RT.ZS", "NY.GDP.MINR.RT.ZS",
    "TX.VAL.FUEL.ZS.UN", "TM.VAL.FUEL.ZS.UN", "TX.VAL.AGRI.ZS.UN", "TM.VAL.AGRI.ZS.UN", "TX.VAL.FOOD.ZS.UN", "TM.VAL.FOOD.ZS.UN",
    "EG.IMP.CONS.ZS", "EG.USE.PCAP.KG.OE", "EG.USE.COMM.GD.PP.KD", "EN.GHG.CO2.MT.CE.AR5", "EN.GHG.CO2.PC.CE.AR5",
    # --- labor, culture, inequality ---
    "SL.TLF.CACT.ZS", "SL.TLF.CACT.MA.ZS", "SL.TLF.CACT.FE.ZS", "SL.UEM.TOTL.ZS", "SL.UEM.1524.ZS", "SL.EMP.TOTL.SP.ZS",
    "SL.EMP.SELF.ZS", "SL.EMP.VULN.ZS", "SL.EMP.WORK.ZS", "SI.POV.GINI", "SI.DST.FRST.20", "SI.DST.05TH.20",
    "SI.DST.10TH.10", "SI.POV.DDAY", "SI.POV.UMIC",
    # --- indebtedness & monetary solvency ---
    "FS.AST.CGOV.GD.ZS", "BN.CAB.XOKA.GD.ZS", "FI.RES.TOTL.MO", "FI.RES.TOTL.CD", "FP.CPI.TOTL.ZG", "NE.CON.GOVT.ZS",
    "GC.DOD.TOTL.GD.ZS", "GC.XPN.TOTL.GD.ZS", "GC.REV.XGRT.GD.ZS", "GC.TAX.TOTL.GD.ZS", "GC.NLD.TOTL.GD.ZS",
    "GC.XPN.INTP.RV.ZS", "GC.XPN.INTP.ZS", "GC.XPN.TRFT.ZS", "DT.DOD.DECT.GN.ZS", "DT.TDS.DECT.EX.ZS",
    "NV.AGR.TOTL.CD"
]
wb_indicators = sorted(set(wb_indicators))

# --- hardened per-country fetch + shared helpers (replaces the old tail of cell 1a) ------------------------------------
WB_TIMEOUT = 40
wb.get_options['timeout'] = WB_TIMEOUT

wb_economies = {e["id"] for e in wb.economy.list()}


def wb_valid_codes(codes, db=2):
    """Drop codes that are not in the database's series list."""
    have = {s["id"] for s in wb.series.list(db=db)}
    bad = sorted(set(codes) - have)
    if bad:
        print(f"  [db {db}] dropping {len(bad)} codes that are not in the series list: {bad}")
    return [c for c in codes if c in have]


def _tall(df_c, batch, iso3, source):
    if df_c is None or df_c.empty:
        return pd.DataFrame(columns=STD_COLS)
    df_c = df_c.reset_index()
    if "series" not in df_c.columns:                         # single-series responses come back without the column
        df_c["series"] = batch[0]
    year_cols = [c for c in df_c.columns if isinstance(c, (int, np.integer)) or (isinstance(c, str) and c.isdigit())]
    out = pd.melt(df_c, id_vars=["series"], value_vars=year_cols, var_name="year", value_name="value").dropna(subset=["value"])
    out["country_code"], out["source"] = iso3, source
    return out.rename(columns={"series": "indicator_code"})[STD_COLS]


def fetch_wb_country(codes, iso3, years, db=2, source="WB", tries=2, chunk=None):
    """One country, many series, one WB database, optionally split in batches of `chunk` codes. Returns a tall frame."""
    codes = list(codes)
    batches = [codes] if not chunk else [codes[i:i + chunk] for i in range(0, len(codes), chunk)]
    parts = []
    for b in batches:
        last, df_c = None, None
        for k in range(tries):
            try:
                kw = dict(economy=iso3, db=db, numericTimeKeys=True)
                if years is not None:
                    kw["time"] = years
                df_c = wb.data.DataFrame(b, **kw)
                last = None
                break
            except Exception as e:
                last = e
                time.sleep(2 * (k + 1))
        if last is not None:
            raise last
        parts.append(_tall(df_c, b, iso3, source))
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=STD_COLS)


def wb_raw_series(iso3, code, db, source, date=None):
    """Plain v2 JSON endpoint, one (country, series) per call. Fallback for databases where wbgapi's batched request is rejected."""
    params = {"format": "json", "per_page": 1000, "source": db}
    if date:
        params["date"] = date
    r = http_get(f"https://api.worldbank.org/v2/country/{iso3}/indicator/{code}", params=params, timeout=WB_TIMEOUT)
    log_fetch(f"WBraw-{db}", r.status_code)
    if r.status_code != 200:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
    js = r.json()
    if not isinstance(js, list) or len(js) < 2 or not js[1]:
        return pd.DataFrame(columns=STD_COLS)
    rows = [(iso3, code, x.get("date"), x.get("value"), source) for x in js[1]]
    return tidy(pd.DataFrame(rows, columns=STD_COLS))


WB_ROUTE = {}                                                 # db -> "raw" once the batched route has been shown not to work


def fetch_wb_country_robust(codes, iso3, years, db, source, chunk=10):
    """wbgapi batches -> same without a time filter -> one raw request per series. Remembers the route per database."""
    err = None
    if WB_ROUTE.get(db) != "raw":
        for yrs in ((years, None) if years is not None else (None,)):
            try:
                return fetch_wb_country(codes, iso3, yrs, db=db, source=source, chunk=chunk, tries=1)
            except Exception as e:
                err = e
    date = f"{min(years)}:{max(years)}" if years is not None else None
    parts, n_err = [], 0
    for c in codes:
        try:
            parts.append(wb_raw_series(iso3, c, db, source, date))
        except Exception as e:
            err, n_err = e, n_err + 1
    if n_err < len(codes):
        if WB_ROUTE.get(db) != "raw":
            print(f"   [db {db}] batched wbgapi request rejected -> using raw per-series requests from now on. First error: {str(err)[:300]}")
        WB_ROUTE[db] = "raw"
        return pd.concat(parts, ignore_index=True)
    raise err


def wb_fetch_many(codes, economies, years, db=2, source="WB", chunk=None, robust=False):
    """Loop over economies; failures get a second pass with a longer timeout. Returns (tall frame, still_failed)."""
    def one(iso3, tries=2):
        if robust:
            return fetch_wb_country_robust(codes, iso3, years, db, source, chunk=chunk or 10)
        return fetch_wb_country(codes, iso3, years, db=db, source=source, tries=tries, chunk=chunk)

    frames, failed = [], []
    for i, iso3 in enumerate(economies, 1):
        try:
            frames.append(one(iso3))
        except Exception as e:
            failed.append(iso3)
            log_fetch(source, "err")
            if len(failed) == 1:
                print(f"   first failure {iso3}: {str(e)[:400]}")      # the old code cut this at 100 chars, which hid the real 400 reason
        if i % 25 == 0 or i == len(economies):
            print(f"  [{i}/{len(economies)}] {source}: processed {iso3}")
    if failed:
        print(f"   retrying {len(failed)} economies with a 90 s timeout: {failed}")
        old = wb.get_options['timeout']
        wb.get_options['timeout'] = 90
        still = []
        for iso3 in failed:
            try:
                frames.append(one(iso3, tries=3))
            except Exception:
                still.append(iso3)
        wb.get_options['timeout'] = old
        failed = still
        if failed:
            print(f"   still failing after retry: {failed}")
    df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=STD_COLS)
    return df, failed


wb_indicators = wb_valid_codes(sorted(set(wb_indicators)), db=2)
wb_targets = [c for c in target_countries if c in wb_economies]
wb_skipped = [c for c in target_countries if c not in wb_economies]
WB_AGGREGATES = [a for a in ("EMU", "WLD") if a in wb_economies]      # Eurozone is one of the report's 24 "countries"; WLD gives world totals for shares
wb_targets += WB_AGGREGATES
print(f"Starting World Bank fetch for {len(wb_indicators)} indicators across {len(wb_targets)} economies "
      f"(skipping non-WB economies: {wb_skipped}; aggregates added: {WB_AGGREGATES}) ...")

df_wb_final, wb_failed = wb_fetch_many(wb_indicators, wb_targets, range(2018, 2026), db=2, source="WB", chunk=60)
df_wb_final = tidy(df_wb_final)
print(f"World Bank: ingested {len(df_wb_final)} observations across {df_wb_final['country_code'].nunique()} economies; failed: {wb_failed}")
wb_missing = sorted(set(wb_indicators) - set(df_wb_final["indicator_code"].unique()))
print(f"{len(wb_missing)} requested WB codes returned no data for 2018-2025: {wb_missing}")

In [ ]:
# 1b. World Bank - other databases (WGI, Doing Business archive, UHC, HCI, population projections)
POP_WANTED = {
    "SP.POP.TOTL", "SP.POP.GROW", "SP.POP.1564.TO", "SP.POP.1564.TO.ZS", "SP.POP.65UP.TO", "SP.POP.65UP.TO.ZS",
    "SP.POP.0014.TO", "SP.POP.0014.TO.ZS", "SP.POP.DPND", "SP.POP.DPND.OL", "SP.POP.DPND.YG",
}


def _wgi_pick(sid, nm):
    return bool(re.fullmatch(r"GOV_WGI_[A-Z]{2}\.(EST|PER\.RNK)", sid))              # decided on the id: the label regex missed the percentile ranks


DB_SCORE_RX = re.compile(r"^(IC\.CNST\.PRMT\.DFRN\.DB\d+|IC\.REG\.DFRN.*|PROT\.MINOR\.INV\.DFRN\.DB\d+|ENF\.CONT\.COEN\.DB\d+\.DFRN"
                         r"|IC\.CRED\.ACC\.LGL\.RGHT\.\d+\.XD\.DB\d+\.DFRN|IC\.BUS\.EASE\.DFRN\.DB\d+)$")

DB_SPECS = {
    3:  dict(label="WGI", expect="Governance", source="WB-WGI", years=range(2015, 2026), pick=_wgi_pick),
    1:  dict(label="DoingBusiness", expect="Doing Business", source="WB-DB", years=range(2004, 2022),
             pick=lambda sid, nm: bool(DB_SCORE_RX.match(sid)), chunk=10, robust=True),
    58: dict(label="UHC", expect="Universal Health", source="WB-UHC", years=range(2000, 2026), pick=lambda sid, nm: True),
    63: dict(label="HCI", expect="Human Capital", source="WB-HCI", years=range(2010, 2026), pick=lambda sid, nm: True),
    40: dict(label="PopProj", expect="Population", source="WB-POP", years=range(2015, 2041), pick=lambda sid, nm: sid in POP_WANTED),
}


def wb_db_name(db):
    try:
        return str(wb.source.get(db)["name"])
    except Exception:
        try:
            return next(str(s["name"]) for s in wb.source.list() if int(s["id"]) == int(db))
        except Exception:
            return ""


wb_other_frames, wb_series_names = [], {}
for db, spec in DB_SPECS.items():
    name = wb_db_name(db)
    if spec["expect"].lower() not in name.lower():
        print(f"[db {db}] expected '{spec['expect']}' but found '{name}' -> skipped. Run wb.source.info() and fix DB_SPECS.")
        continue
    series = [(s["id"], s["value"]) for s in wb.series.list(db=db)]
    chosen = [sid for sid, nm in series if spec["pick"](sid, nm)]
    if db == 1 and not chosen:                                   # id pattern did not match anything -> every distance-to-frontier series
        chosen = [sid for sid, _ in series if "DFRN" in sid]
    wb_series_names.update({sid: nm for sid, nm in series if sid in chosen})
    print(f"[db {db}] {name}: {len(chosen)}/{len(series)} series selected, {len(wb_targets)} economies, "
          f"years {spec['years'][0]}-{spec['years'][-1]}")
    if db in (1, 3):
        print("   series:", sorted(chosen))
    df_db, failed = wb_fetch_many(chosen, wb_targets, spec["years"], db=db, source=spec["source"],
                                  chunk=spec.get("chunk"), robust=spec.get("robust", False))
    got = set(df_db["indicator_code"].unique()) if not df_db.empty else set()
    empty = sorted(set(chosen) - got)
    if db == 3 and not df_db.empty:                              # GOV_WGI_CC.EST -> CC.EST, so the tree's "CC.EST" / "RL.PER.RNK" match
        df_db["indicator_code"] = df_db["indicator_code"].str.replace(r"^GOV_WGI_", "", regex=True)
    if db == 40 and not df_db.empty:                             # forward-looking years must never count as "recent actuals"
        df_db["year"] = pd.to_numeric(df_db["year"])
        fwd = df_db["year"] >= PROJ_FROM
        df_db.loc[fwd, "indicator_code"] = "PROJ." + df_db.loc[fwd, "indicator_code"]
        df_db["is_projection"] = fwd
    wb_other_frames.append(df_db)
    print(f"   -> {len(df_db)} rows; {len(empty)} selected series had no data in window" + (f": {empty[:8]}" if empty else "")
          + (f"; economies that failed: {failed}" if failed else ""))

df_wb_other = tidy(pd.concat(wb_other_frames, ignore_index=True)) if wb_other_frames else pd.DataFrame(columns=STD_COLS)
print("WB other databases:", len(df_wb_other), "rows,", df_wb_other["indicator_code"].nunique(), "series")

In [ ]:
# 1c. World Bank - long history (volatility, capital stock via perpetual inventory, working-age growth)
WB_LONG_CODES = [
    "NY.GDP.MKTP.KD.ZG", "NY.GDP.MKTP.KD", "NY.GDP.PCAP.KD", "FP.CPI.TOTL.ZG", "NY.GDP.DEFL.KD.ZG",
    "NE.GDI.FTOT.ZS", "NE.GDI.FTOT.KD", "NY.GNS.ICTR.ZS", "SL.GDP.PCAP.EM.KD", "SL.UEM.TOTL.ZS",
    "SP.POP.1564.TO", "SP.POP.DPND", "SP.POP.TOTL", "FS.AST.PRVT.GD.ZS", "GC.DOD.TOTL.GD.ZS",
]
df_wb_long, wb_long_failed = wb_fetch_many(wb_valid_codes(WB_LONG_CODES, db=2), wb_targets, range(1990, 2026), db=2, source="WB-LONG")
df_wb_long = tidy(df_wb_long)
print("WB long history:", len(df_wb_long), "rows; failed:", wb_long_failed)

In [ ]:
# 2. World Bank EdStats (db=12): Barro-Lee attainment (#16) and PISA
EDSTATS_CODES = [
    "BAR.SCHL.25UP", "BAR.NOED.25UP.ZS", "BAR.PRM.CMPT.25UP.ZS", "BAR.SEC.CMPT.25UP.ZS", "BAR.TER.CMPT.25UP.ZS",
    "LO.PISA.MAT", "LO.PISA.REA", "LO.PISA.SCI",
]
df_wb_edstats, ed_failed = wb_fetch_many(wb_valid_codes(EDSTATS_CODES, db=12), wb_targets, range(2000, 2026), db=12, source="WB-EDSTATS")
df_wb_edstats = tidy(df_wb_edstats)
if not df_wb_edstats.empty:
    print(f"WB EdStats: {len(df_wb_edstats)} observations, {df_wb_edstats['country_code'].nunique()} countries, "
          f"{df_wb_edstats['indicator_code'].nunique()}/{len(EDSTATS_CODES)} series returned data; failed: {ed_failed}")
    print("No data for:", sorted(set(EDSTATS_CODES) - set(df_wb_edstats['indicator_code'].unique())))
else:
    print("WB EdStats: nothing returned (check codes / db=12).")

In [ ]:
# 3. IMF WEO
# The API ignores startPeriod/endPeriod (full 1980-2031 history comes back), so rows are clipped here.
# The response has no LATEST_ACTUAL column, so projections fall back to year >= PROJ_FROM.
IMF_START, IMF_END = 2000, 2030
imf_series = [
    "GGXWDG_NGDP", "GGXCNL_NGDP", "NGDP_RPCH", "BCA_NGDPD", "PCPIPCH", "LUR",
    "NGDPD", "NGDPDPC", "PPPGDP", "PPPPC", "PPPSH", "LP",
    "NGSD_NGDP", "NID_NGDP",
    "GGR_NGDP", "GGX_NGDP", "GGXONLB_NGDP", "GGXWDN_NGDP",
    "BCA", "TXG_RPCH", "TMG_RPCH", "NGAP_NPGDP",
]
imf_indicators = "+".join(imf_series)
IMF_API = "https://api.imf.org/external/sdmx/3.0/data/dataflow"


def parse_imf_csv(text):
    df_c = pd.read_csv(io.StringIO(text))
    geo_col = "COUNTRY" if "COUNTRY" in df_c.columns else "REF_AREA"
    if df_c.empty or "OBS_VALUE" not in df_c.columns or geo_col not in df_c.columns:
        return pd.DataFrame(columns=STD_COLS), list(df_c.columns)
    sub = df_c.rename(columns={geo_col: "country_code", "INDICATOR": "indicator_code",
                               "TIME_PERIOD": "year", "OBS_VALUE": "value"}).copy()
    sub["year"] = pd.to_numeric(sub["year"], errors="coerce")
    la_col = next((c for c in df_c.columns if "LATEST_ACTUAL" in c.upper()), None)
    if la_col:
        la = pd.to_numeric(sub[la_col], errors="coerce")
        sub["is_projection"] = np.where(la.notna(), sub["year"] > la, sub["year"] >= PROJ_FROM)
    else:
        sub["is_projection"] = sub["year"] >= PROJ_FROM
    sub["indicator_code"] = "IMF." + sub["indicator_code"].astype(str)
    sub["source"] = "IMF"
    sub = sub[sub["year"].between(IMF_START, IMF_END)]
    return tidy(sub[["country_code", "indicator_code", "year", "value", "source", "is_projection"]]), list(df_c.columns)


print(f"Starting IMF SDMX 3.0 loop across {len(target_countries)} countries...")
imf_results, imf_bad, imf_cols = [], [], None
for i, iso3 in enumerate(target_countries, 1):
    url = f"{IMF_API}/IMF.RES/WEO/~/{iso3}.{imf_indicators}.A"
    try:
        resp = http_get(url, params={"startPeriod": str(IMF_START), "endPeriod": str(IMF_END)},
                        headers={"Accept": "text/csv"}, timeout=30)
        log_fetch("IMF", resp.status_code)
        if resp.status_code == 200 and resp.text.strip():
            part, cols = parse_imf_csv(resp.text)
            if imf_cols is None:
                imf_cols = cols
                print("  IMF response columns:", cols)
            if not part.empty:
                imf_results.append(part)
        elif resp.status_code != 404:
            imf_bad.append((iso3, resp.status_code))
    except Exception as e:
        print(f"  [{i}/{len(target_countries)}] Failed {iso3}: {str(e)[:100]}")
    if i % 25 == 0 or i == len(target_countries):
        print(f"  [{i}/{len(target_countries)}] Processed {iso3}")

df_imf_final = pd.concat(imf_results, ignore_index=True) if imf_results else pd.DataFrame(columns=STD_COLS)
print(f"IMF WEO: {len(df_imf_final)} rows, {df_imf_final['country_code'].nunique()} countries, "
      f"{df_imf_final['indicator_code'].nunique()} series, {int(df_imf_final['is_projection'].sum())} projection rows, "
      f"{int(df_imf_final.duplicated(['country_code', 'indicator_code', 'year']).sum())} duplicate keys.")
if imf_bad:
    print("Non-200/404 responses:", imf_bad[:10])

In [ ]:
# 3b. IMF COFER - currency shares of world allocated reserves -> "reserve currency status" (report #10, 12% of strength)
# COFER is published only for the world aggregate (G001) by currency; each currency is assigned to its issuer(s).
# Non-issuers get a structural 0. Key order assumed: COUNTRY.INDICATOR.FXR_CURRENCY.TYPE_OF_TRANSFORMATION.FREQUENCY (check the printout).
EURO_AREA = ["AUT", "BEL", "BGR", "HRV", "CYP", "EST", "FIN", "FRA", "DEU", "GRC", "IRL", "ITA", "LVA", "LTU", "LUX",
             "MLT", "NLD", "PRT", "SVK", "SVN", "ESP", "EMU"]          # BGR joined in 2026; the share is applied to all years for simplicity
COFER_ISSUERS = {"USD": ["USA"], "EUR": EURO_AREA, "JPY": ["JPN"], "GBP": ["GBR"], "CNY": ["CHN"],
                 "AUD": ["AUS"], "CAD": ["CAN"], "CHF": ["CHE", "LIE"]}


def cofer_fetch():
    for key in ("G001.AFXRA.*.SHRO_PT.Q", "G001.*.*.*.Q"):
        r = http_get(f"{IMF_API}/IMF.STA/COFER/~/{key}", headers={"Accept": "text/csv"}, timeout=60)
        log_fetch("IMF-COFER", r.status_code)
        if r.status_code == 200 and r.text.strip():
            return pd.read_csv(io.StringIO(r.text))
    return None


def parse_cofer(raw, countries):
    cols = {c.upper(): c for c in raw.columns}
    ccy_c = next((cols[c] for c in cols if "CURRENCY" in c), None)
    tr_c = next((cols[c] for c in cols if "TRANSFORMATION" in c), None)
    ind_c = cols.get("INDICATOR")
    if ccy_c is None or "OBS_VALUE" not in raw.columns:
        return pd.DataFrame(columns=STD_COLS)
    d = raw.copy()
    if tr_c is not None:
        d = d[d[tr_c].astype(str).str.contains("SHR", case=False)]
    if ind_c is not None and "AFXRA" in set(d[ind_c].astype(str)):
        d = d[d[ind_c].astype(str) == "AFXRA"]
    d["ccy"] = d[ccy_c].astype(str).str.replace("CI_", "", regex=False).str.upper()
    d = d[d["ccy"].isin(COFER_ISSUERS)]
    d["year"] = pd.to_numeric(d["TIME_PERIOD"].astype(str).str[:4], errors="coerce")
    d["value"] = pd.to_numeric(d["OBS_VALUE"], errors="coerce")
    d = d.dropna(subset=["year", "value"]).sort_values("TIME_PERIOD").groupby(["ccy", "year"], as_index=False).last()   # last quarter of the year
    rows = []
    for _, r in d.iterrows():
        for c in COFER_ISSUERS[r["ccy"]]:
            rows.append((c, "IMF.COFER_SHARE", r["year"], r["value"], "IMF-COFER"))
    out = pd.DataFrame(rows, columns=STD_COLS)
    issuers = {c for lst in COFER_ISSUERS.values() for c in lst}
    for c in countries:
        if c not in issuers:
            for y in sorted(d["year"].unique()):
                rows.append((c, "IMF.COFER_SHARE", y, 0.0, "IMF-COFER"))
    return tidy(pd.DataFrame(rows, columns=STD_COLS))


try:
    cofer_raw = cofer_fetch()
    if cofer_raw is not None:
        print("COFER columns:", list(cofer_raw.columns))
    df_cofer = parse_cofer(cofer_raw, target_countries) if cofer_raw is not None else pd.DataFrame(columns=STD_COLS)
except Exception as e:
    print("COFER failed:", str(e)[:150])
    df_cofer = pd.DataFrame(columns=STD_COLS)
print(f"IMF COFER: {len(df_cofer)} rows, issuers with data: "
      f"{sorted(df_cofer[df_cofer['value'] > 0]['country_code'].unique())[:12]}")

In [ ]:
# 4. US EIA International
# The facet route only lists a subset of products (no total-energy, electricity, coal or gas), so products are discovered by
# probing product ids one by one for the USA and reading the productName from the first row. Then: one request per (country, product).
EIA_KEY = "kJpbTkcdzNctTcTMKHJF9bWdkwbwmqHWJFfRnKGL"
EIA_BASE = "https://api.eia.gov/v2/"
EIA_START, EIA_END = 2005, 2026


def eia_call(route, params, timeout=30):
    r = http_get(EIA_BASE + route, params={"api_key": EIA_KEY, **params}, timeout=timeout)
    log_fetch("EIA", r.status_code)
    r.raise_for_status()
    return r.json().get("response", {})


def eia_probe(ids, iso3="USA"):
    found = {}
    for pid in ids:
        try:
            resp = eia_call("international/data", {"frequency": "annual", "data[0]": "value",
                                                   "facets[productId][]": str(pid), "facets[countryRegionId][]": iso3,
                                                   "start": "2020", "end": "2021", "length": 1})
            rows = resp.get("data", [])
            if rows:
                found[str(pid)] = str(rows[0].get("productName", "?"))
        except Exception:
            pass
        time.sleep(0.03)
    return found


def eia_fetch_rows(iso3, product_id):
    rows, offset = [], 0
    while True:
        resp = eia_call("international/data", {
            "frequency": "annual", "data[0]": "value",
            "facets[productId][]": product_id, "facets[countryRegionId][]": iso3,
            "start": str(EIA_START), "end": str(EIA_END), "length": 5000, "offset": offset,
        })
        batch = resp.get("data", [])
        rows += batch
        try:
            total = int(resp.get("total", len(rows)))
        except (TypeError, ValueError):
            total = len(rows)
        offset += 5000
        if not batch or offset >= total:
            return rows


def parse_eia_rows(rows, iso3, tag, legacy=False):
    cols = ["period", "activityId", "activityName", "unit", "value"]
    if not rows:
        return pd.DataFrame(columns=STD_COLS)
    d = pd.DataFrame(rows)
    for c in cols:
        if c not in d.columns:
            d[c] = np.nan
    d["year"] = pd.to_numeric(d["period"], errors="coerce")
    d["value"] = pd.to_numeric(d["value"], errors="coerce")
    d["activityId"] = d["activityId"].astype(str)
    act = d["activityName"].fillna("ACT" + d["activityId"])
    d["indicator_code"] = "EIA." + tag + "." + act.map(slug) + "." + d["unit"].fillna("NA").map(slug)
    d["country_code"] = iso3
    out = [tidy(d[["country_code", "indicator_code", "year", "value"]], "EIA")]
    if legacy:                                                        # total primary energy: production (activity 1) & consumption (2), QBTU
        q = d[(d["unit"] == "QBTU") & d["activityId"].isin(["1", "2"])]
        pv = q.pivot_table(index="year", columns="activityId", values="value", aggfunc="first")
        if "1" in pv:
            out.append(tidy(pd.DataFrame({"country_code": iso3, "indicator_code": "EIA.ENERGY.PRODUCTION_QBTU",
                                          "year": pv.index, "value": pv["1"].values}), "EIA"))
        if "1" in pv and "2" in pv:
            ok = pv["2"] > 0
            out.append(tidy(pd.DataFrame({"country_code": iso3, "indicator_code": "EIA.ENERGY.SELF_SUFF",
                                          "year": pv.index[ok], "value": ((pv["1"] - pv["2"]) / pv["2"])[ok].values}), "EIA"))
    return pd.concat(out, ignore_index=True)


EIA_PATTERNS = {"ELECTRICITY": r"electricity", "COAL": r"^coal", "NATGAS": r"natural gas", "NUCLEAR": r"nuclear",
                "RENEWABLES": r"renewable", "OIL": r"^petroleum and other liquids$|^total petroleum"}
eia_prod_names = eia_probe(range(1, 131))
EIA_PRODUCTS = {"44": ("TOTAL_ENERGY", True)}
_used_tags = {"TOTAL_ENERGY"}
for pid, nm in eia_prod_names.items():
    if pid not in EIA_PRODUCTS and len(EIA_PRODUCTS) < 14 and any(re.search(p, nm, re.I) for p in EIA_PATTERNS.values()):
        tag = slug(nm)[:30]
        if tag in _used_tags:                                    # e.g. product 7 "Coal" vs product 30 "Coal"
            tag = f"{tag}_{pid}"
        _used_tags.add(tag)
        EIA_PRODUCTS[pid] = (tag, False)
print("EIA products found by probing:", eia_prod_names)
print("EIA products used:", {pid: (v[0], eia_prod_names.get(pid, "?")) for pid, v in EIA_PRODUCTS.items()})

eia_frames = []
print(f"Starting EIA loop: {len(target_countries)} countries x {len(EIA_PRODUCTS)} products...")
for i, iso3 in enumerate(target_countries, 1):
    for pid, (tag, legacy) in EIA_PRODUCTS.items():
        try:
            part = parse_eia_rows(eia_fetch_rows(iso3, pid), iso3, tag, legacy)
            if not part.empty:
                eia_frames.append(part)
        except Exception:
            log_fetch("EIA", "err")
        time.sleep(0.05)
    if i % 25 == 0 or i == len(target_countries):
        print(f"  [{i}/{len(target_countries)}] Processed {iso3}")

df_eia_final = pd.concat(eia_frames, ignore_index=True) if eia_frames else pd.DataFrame(columns=STD_COLS)
print(f"EIA: {len(df_eia_final)} rows, {df_eia_final['country_code'].nunique()} countries, {df_eia_final['indicator_code'].nunique()} series.")

In [ ]:
# 5. V-Dem
url_vdem = "https://www.v-dem.net/media/datasets/V-Dem-CY-FullOthers-v16_csv.zip"
vdem_indicators = [
    "v2cltrnslw", "v2excrptps", "v2exthftps", "v2clrspct", "v2x_rule", "v2cacamps", "v2caviol",       # report #46, #51, #61 + conflict
    "v2x_corr", "v2x_pubcorr", "v2x_execorr", "v2x_jucon", "v2xcl_rol", "v2xcl_prpty",                # corruption / rule of law / property rights
    "v2x_polyarchy", "v2x_libdem", "v2x_accountability",                                              # democracy quality
    "v2x_clphy", "v2cagenmob", "v2cademmob", "v2smpolsoc", "v2smpolhate",                             # internal conflict
    "v2xeg_eqdr", "v2xeg_eqprotec", "v2x_egaldem",                                                    # wealth / opportunity gaps
]
print("Streaming V-Dem v16 Full+Others archive into memory...")
res = http_get(url_vdem, timeout=180, tries=2)
res.raise_for_status()

want = {"country_text_id", "year", *vdem_indicators}
with zipfile.ZipFile(io.BytesIO(res.content)) as z:
    member = next(n for n in z.namelist() if n.lower().endswith(".csv") and "cy-full" in n.lower())
    with z.open(member) as f:
        df_vdem_master = pd.read_csv(f, usecols=lambda c: c in want, low_memory=False)

vdem_present = [c for c in vdem_indicators if c in df_vdem_master.columns]
print("V-Dem columns not in file:", sorted(set(vdem_indicators) - set(vdem_present)))

df_vdem_master = df_vdem_master[df_vdem_master["year"] >= 2018]
by_country = {k: g for k, g in df_vdem_master.groupby("country_text_id")}

vdem_rows = []
for iso3 in target_countries:                      # per-country slice of the single download
    g = by_country.get(iso3)
    if g is None:
        continue
    m = pd.melt(g, id_vars=["country_text_id", "year"], value_vars=vdem_present,
                var_name="indicator_code", value_name="value").dropna()
    vdem_rows.append(m)

df_vdem_final = pd.concat(vdem_rows, ignore_index=True).rename(columns={"country_text_id": "country_code"})
df_vdem_final["indicator_code"] = "VDEM." + df_vdem_final["indicator_code"]
df_vdem_final["source"] = "VDEM"
print(f"V-Dem: {len(vdem_present)} series ({len(df_vdem_final)} rows) across {df_vdem_final['country_code'].nunique()} countries.")

In [ ]:
# 6. ND-GAIN
url_gain = "https://gain.nd.edu/assets/647440/ndgain_countryindex_2026.zip"
print("Streaming ND-GAIN 2026 archive into memory...")
res = http_get(url_gain, timeout=60, tries=2)
res.raise_for_status()


def ndgain_code(name):
    parts = name.replace("\\", "/").split("/")
    stem = parts[-1].rsplit(".", 1)[0]
    parent = parts[-2] if len(parts) > 1 else ""
    if stem.lower() in ("gain", "readiness", "vulnerability"):
        return f"NDGAIN.{stem.upper()}"
    return f"NDGAIN.{parent.upper()}.{stem.upper()}"


ndgain_frames = []
with zipfile.ZipFile(io.BytesIO(res.content)) as z:
    for name in z.namelist():
        low = name.lower()
        if not low.endswith(".csv") or any(t in low for t in ("rank", "delta")):
            continue
        try:
            df_m = pd.read_csv(z.open(name))
        except Exception:
            continue
        if "ISO3" not in df_m.columns:
            continue
        year_cols = [c for c in df_m.columns if str(c).isdigit() and int(c) >= 2018]
        if not year_cols:
            continue
        code = ndgain_code(name)
        for iso3 in target_countries:               # per-country slice
            row = df_m[df_m["ISO3"] == iso3]
            if row.empty:
                continue
            m = pd.melt(row, id_vars=["ISO3"], value_vars=year_cols, var_name="year", value_name="value").dropna()
            m["indicator_code"] = code
            m["source"] = "NDGAIN"
            ndgain_frames.append(m.rename(columns={"ISO3": "country_code"})[
                ["country_code", "indicator_code", "year", "value", "source"]])

df_ndgain_final = (pd.concat(ndgain_frames, ignore_index=True)
                   .drop_duplicates(["country_code", "indicator_code", "year"])) if ndgain_frames else pd.DataFrame(columns=STD_COLS)
print(f"ND-GAIN: {df_ndgain_final['indicator_code'].nunique()} series ({len(df_ndgain_final)} rows) "
      f"across {df_ndgain_final['country_code'].nunique()} countries.")

In [ ]:
# 7. Our World in Data (per-country filtered CSVs with high-frequency date handling)
OWID_CHARTS = {
    # Existing gauges
    "nuclear-warhead-stockpiles":                    "FAS.NUCLEAR_STOCKPILE",
    "happiness-cantril-ladder":                      "WHR.CANTRIL_LADDER",
    "annual-working-hours-per-worker":               "PWT.HOURS_PER_WORKER",
    "annual-co2-emissions-per-country":              "OWID.CO2_TOTAL_T",
    "co-emissions-per-capita":                       "OWID.CO2_PER_CAPITA_T",
    "electricity-generation":                        "OWID.ELECTRICITY_GEN_TWH",
    "primary-energy-cons":                           "OWID.PRIMARY_ENERGY_TWH",
    "oil-production-by-country":                     "OWID.OIL_PRODUCTION_TWH",
    "gas-production-by-country":                     "OWID.GAS_PRODUCTION_TWH",
    "coal-production-by-country":                    "OWID.COAL_PRODUCTION_TWH",
    "ti-corruption-perception-index":                "TI.CPI",
    "excess-mortality-p-scores-average-baseline":     "COVID.EXCESS_MORTALITY_PSCORE",
    
    # Missing Dalio Report Sources implemented via OWID:
    # Report #59: Top 1% Wealth Share (World Inequality Database)
    "wealth-share-richest":                          "WID.TOP1_WEALTH",
    # Report #58: Top 10% Pre-tax Income Share (WID)
    "share-of-the-top-10-in-pre-tax-national-income": "WID.TOP10_INCOME",
    # Report #31: Armed Conflict Deaths (UCDP)
    "deaths-in-armed-conflicts-by-country":          "UCDP.INTERNAL_CONFLICT_DEATHS",
    # Culture / Labor: Labor share of GDP
    "labor-share-of-gdp":                            "ILO.LABOR_SHARE"
}

OWID_SPARSE_OK = {"nuclear-warhead-stockpiles", "deaths-in-armed-conflicts-by-country"}
OWID_WINDOWS = {"excess-mortality-p-scores-projected-baseline": ("2020-01-01", "2022-12-31")}   # extra single value per country
OWID_MIN_YEAR = 2005
OWID_BASE = "https://ourworldindata.org/grapher/{slug}.csv"
OWID_SKIP_COL = re.compile(r"original year|annotation|world region|^entity$|^code$|^year$|^day$", re.I)


def parse_owid_csv(text, iso3, code, window=None):
    """Annual (`Year`) or dated (`Day`) OWID chart -> tidy rows. Dated charts become annual means (+ one window mean)."""
    d = pd.read_csv(io.StringIO(text))
    if d.empty:
        return pd.DataFrame(columns=STD_COLS), []
    if "Year" in d.columns:
        dates, yr = None, pd.to_numeric(d["Year"], errors="coerce")
    elif "Day" in d.columns:
        dates = pd.to_datetime(d["Day"], errors="coerce")
        yr = dates.dt.year
    else:
        return pd.DataFrame(columns=STD_COLS), []
    val_cols = [c for c in d.columns if not OWID_SKIP_COL.search(str(c)) and pd.to_numeric(d[c], errors="coerce").notna().any()]
    frames = []
    for j, c in enumerate(val_cols):
        cc_ = code if j == 0 else f"{code}.{slug(c)}"            # first value column keeps the base code
        v = pd.to_numeric(d[c], errors="coerce")
        f = pd.DataFrame({"year": yr, "value": v}).dropna()
        if dates is not None:
            f = f.groupby("year", as_index=False)["value"].mean()
            if window and j == 0:
                w = v[(dates >= pd.Timestamp(window[0])) & (dates <= pd.Timestamp(window[1]))].dropna()
                if len(w):
                    frames.append(pd.DataFrame({"country_code": iso3, "indicator_code": f"{code}_{window[0][:4]}_{window[1][:4]}_MEAN",
                                                "year": int(window[1][:4]), "value": w.mean()}, index=[0]))
        frames.append(f.assign(country_code=iso3, indicator_code=cc_))
    if not frames:
        return pd.DataFrame(columns=STD_COLS), val_cols
    out = tidy(pd.concat(frames, ignore_index=True), "OWID")
    return out[out["year"] >= OWID_MIN_YEAR], val_cols


owid_rows, owid_stat, owid_dead, owid_cols = [], {s: defaultdict(int) for s in OWID_CHARTS}, set(), {}
print(f"Starting OWID loop across {len(target_countries)} countries x {len(OWID_CHARTS)} charts...")
for i, iso3 in enumerate(target_countries, 1):
    for chart, code in OWID_CHARTS.items():
        if chart in owid_dead:
            continue
        try:
            r = http_get(OWID_BASE.format(slug=chart), params={"csvType": "filtered", "country": f"~{iso3}"},
                         headers={"User-Agent": "Our World In Data data fetch/1.0"}, timeout=25)
            owid_stat[chart][r.status_code] += 1
            if r.status_code == 200 and r.text.strip():
                part, val_cols = parse_owid_csv(r.text, iso3, code, OWID_WINDOWS.get(chart))
                if chart not in owid_cols and val_cols:
                    owid_cols[chart] = val_cols
                if not part.empty:
                    owid_rows.append(part)
                    owid_stat[chart]["rows_ok"] += 1
            elif (chart not in OWID_SPARSE_OK and sum(v for k, v in owid_stat[chart].items() if k != "rows_ok") >= 5
                  and owid_stat[chart].get(200, 0) == 0):
                owid_dead.add(chart)
        except Exception:
            owid_stat[chart]["exc"] += 1
        time.sleep(0.15)
    if i % 25 == 0 or i == len(target_countries):
        print(f"  [{i}/{len(target_countries)}] Processed {iso3}")

df_owid_final = pd.concat(owid_rows, ignore_index=True) if owid_rows else pd.DataFrame(columns=STD_COLS)
print(f"OWID: {len(df_owid_final)} rows")
for chart, st in owid_stat.items():
    flag = "  <-- no parsable data at all (dead slug, or a new column layout)" if st.get("rows_ok", 0) == 0 and chart not in OWID_SPARSE_OK else ""
    http_codes = {k: v for k, v in st.items() if k != "rows_ok"}
    print(f"  {chart:<46} countries_with_data={st.get('rows_ok', 0):<4} http={http_codes}{flag}")
    if len(owid_cols.get(chart, [])) > 1:
        print(f"      multiple value columns -> first = base code: {owid_cols[chart]}")

In [ ]:
# 8. ILOSTAT - hours worked, participation, unemployment (+ catalogue-discovered extras)
# Per-country requests; if the first five requests of an indicator all fail, the bulk file is downloaded once and sliced per country.
ILO_DATA = "https://rplumber.ilo.org/data/indicator/"
ILO_BULK = "https://www.ilo.org/ilostat-files/WEB_bulk_download/indicator/{id}.csv.gz"
ILO_TOC = "https://rplumber.ilo.org/metadata/toc/indicator/"
ILO_IDS = {                                                    # id -> (code, mode)   mode: total (age 15+ aggregate) | suffix (keep breakdowns)
    "HOW_2EMP_SEX_NB_A":     ("ILO.HOURS_WEEKLY_MODELLED", "total"),
    "EAP_2WAP_SEX_AGE_RT_A": ("ILO.LFPR_MODELLED", "total"),
    "UNE_2EAP_SEX_AGE_RT_A": ("ILO.UNEMP_RATE_MODELLED", "total"),
    "EMP_2WAP_SEX_AGE_RT_A": ("ILO.EMP_POP_RATIO_MODELLED", "total"),
}
ILO_DISCOVER = {                                               # code -> regex on the catalogue label (ids found at run time)
    "ILO.COLLECTIVE_BARGAINING_COVERAGE": r"collective bargaining coverage",
    "ILO.MIN_WAGE": r"minimum wage",
    "ILO.UNEMP_LONGTERM_SHARE": r"long-term unemployment|unemployment by .*duration",
    "ILO.STRIKES_LOCKOUTS": r"strikes and lockouts|days not worked",
}
ILO_CLASSIF_OK = {"AGE_YTHADULT_YGE15", "AGE_AGGREGATE_TOTAL", "ECO_AGGREGATE_TOTAL", "OCU_SKILL_TOTAL", "AGE_10YRBANDS_TOTAL"}
ILO_CLASSIF_OK |= {"AGE_AGGREGATE_YGE15", "ECO_ISIC4_TOTAL", "ECO_ISIC3_TOTAL", "ECO_SECTOR_TOTAL", "ECO_DETAILS_TOTAL"}
ILO_MIN_YEAR = 2000
ilo_unmatched = defaultdict(set)


def parse_ilo_df(d, iso3, ilo_id, code, mode="total"):
    d = d.copy()
    d.columns = [str(c).lower() for c in d.columns]
    if not {"obs_value", "time"} <= set(d.columns):
        return pd.DataFrame(columns=STD_COLS)
    if "ref_area" in d.columns:
        d = d[d["ref_area"].astype(str).str.upper() == iso3]
    if "sex" in d.columns:
        d = d[d["sex"].astype(str).isin(["SEX_T", "Total", "T"])]
    d["indicator_code"] = code
    if mode == "total":
        if "classif1" in d.columns:
            ok = d["classif1"].isna() | d["classif1"].astype(str).isin(ILO_CLASSIF_OK)
            ilo_unmatched[ilo_id] |= set(d.loc[~ok, "classif1"].dropna().astype(str).unique())
            d = d[ok]
        if "classif2" in d.columns:
            d = d[d["classif2"].isna()]
    else:
        for c in ("classif1", "classif2"):
            if c in d.columns:
                d["indicator_code"] = d["indicator_code"] + np.where(d[c].notna(), "." + d[c].astype(str).map(slug), "")
    d = d.assign(country_code=iso3, year=d["time"], value=d["obs_value"])
    out = tidy(d[["country_code", "indicator_code", "year", "value"]], "ILO")
    return out[out["year"] >= ILO_MIN_YEAR].drop_duplicates(["country_code", "indicator_code", "year"])


def ilo_get_country(ilo_id, iso3):
    r = http_get(ILO_DATA, params={"id": ilo_id, "ref_area": iso3, "timefrom": ILO_MIN_YEAR, "format": ".csv"}, timeout=40)
    log_fetch("ILO", r.status_code)
    if r.status_code != 200 or not r.text.strip():
        return None
    try:
        return pd.read_csv(io.StringIO(r.text))
    except Exception:
        return None


def ilo_load_bulk(ilo_id):
    try:
        r = http_get(ILO_BULK.format(id=ilo_id), timeout=180, tries=2)
        log_fetch("ILO-bulk", r.status_code)
        if r.status_code == 200:
            b = pd.read_csv(io.BytesIO(r.content), compression="gzip", low_memory=False)
            b.columns = [str(c).lower() for c in b.columns]
            return b
    except Exception:
        pass
    return None


def ilo_discover(patterns):
    """Search the ILOSTAT indicator catalogue; returns {id: (code, 'suffix')} for annual indicators whose label matches."""
    found = {}
    try:
        toc = pd.read_csv(io.StringIO(http_get(ILO_TOC, params={"lang": "en", "format": ".csv"}, timeout=40).text))
    except Exception as e:
        print("  ILOSTAT catalogue not readable:", str(e)[:100])
        return found
    id_col = next((c for c in toc.columns if re.fullmatch(r"(?i)(indicator|id|indicator_code)", str(c))), toc.columns[0])
    lab_cols = [c for c in toc.columns if "label" in str(c).lower() or "title" in str(c).lower()] or [c for c in toc.columns if c != id_col]
    # blank cells arrive as NaN floats (pandas 3 keeps them through astype(str)), so fill before joining
    text = toc[lab_cols].fillna("").astype(str).apply(lambda r: " ".join(r), axis=1)
    is_annual = toc[id_col].fillna("").astype(str).str.endswith("_A")
    for code, pat in patterns.items():
        hit = toc[text.str.contains(pat, case=False, regex=True, na=False) & is_annual]
        for ilo_id in hit[id_col].astype(str).head(2):
            found[ilo_id] = (code, "suffix")
    print("  ILOSTAT catalogue columns:", list(toc.columns), "| discovered:", {k: v[0] for k, v in found.items()})
    return found


ILO_IDS.update(ilo_discover(ILO_DISCOVER))
for _k, (_code, _m) in list(ILO_IDS.items()):
    if _code.startswith("ILO.STRIKES"):
        ILO_IDS[_k] = (_code, "total")
print("ILO indicators:", {k: v[0] for k, v in ILO_IDS.items()})

ilo_frames, ilo_failed = [], []
for ilo_id, (code, mode) in ILO_IDS.items():
    n_ok, tried, bulk = 0, 0, None
    for iso3 in target_countries:
        d = ilo_get_country(ilo_id, iso3)
        tried += 1
        if d is not None and len(d):
            part = parse_ilo_df(d, iso3, ilo_id, code, mode)
            if not part.empty:
                ilo_frames.append(part)
                n_ok += 1
        elif d is None and n_ok == 0 and tried >= 5:            # per-country endpoint looks broken -> bulk file, sliced per country
            bulk = ilo_load_bulk(ilo_id)
            break
        time.sleep(0.05)
    if bulk is not None and "ref_area" in bulk.columns:
        for iso3 in target_countries:
            d = bulk[bulk["ref_area"].astype(str).str.upper() == iso3]
            if len(d):
                part = parse_ilo_df(d, iso3, ilo_id, code, mode)
                if not part.empty:
                    ilo_frames.append(part)
                    n_ok += 1
    print(f"  {ilo_id:<32} -> {code:<36} countries with data: {n_ok}" + ("   (via bulk file)" if bulk is not None else ""))
    if n_ok == 0:
        ilo_failed.append(ilo_id)

df_ilo_final = pd.concat(ilo_frames, ignore_index=True) if ilo_frames else pd.DataFrame(columns=STD_COLS)
for code in {c for c, m in ILO_IDS.values() if m == "suffix"}:                   # cap breakdown explosion: 12 most frequent variants per base code
    mask = df_ilo_final["indicator_code"].str.startswith(code)
    keep = df_ilo_final[mask]["indicator_code"].value_counts().head(12).index
    df_ilo_final = df_ilo_final[~mask | df_ilo_final["indicator_code"].isin(keep)]
print(f"ILOSTAT: {len(df_ilo_final)} rows, {df_ilo_final['indicator_code'].nunique()} series")
if any(ilo_unmatched.values()):
    print("classif1 values skipped (extend ILO_CLASSIF_OK if one of them is the total):",
          {k: sorted(v)[:6] for k, v in ilo_unmatched.items() if v})
if ilo_failed:
    print("No data for:", ilo_failed)

In [ ]:
# 9. BIS bulk files (one download per dataset, sliced per country)
# WS_TC (credit to the non-financial sector, % of GDP, by borrower sector) -> debt levels and flow
# WS_CBPOL (central-bank policy rates) -> monetary policy
# Column headers/values come as "CODE:Label"; both are reduced to the code. Set BIS_INSPECT to print the dimensions of the
# datasets still to be wired up for reserve-currency / financial-center items (FX turnover, cross-border claims, debt securities).
BIS_BASE = "https://data.bis.org/static/bulk/"
BIS_INSPECT = []                  # e.g. ["WS_DER_OTC_TOV", "WS_LBS_D_PUB", "WS_DEBT_SEC2_PUB"]


def bis_download(stem):
    r = http_get(f"{BIS_BASE}{stem}_csv_flat.zip", timeout=300, tries=2)
    log_fetch("BIS", r.status_code)
    if r.status_code != 200:
        return None
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        name = next(n for n in z.namelist() if n.lower().endswith(".csv"))
        df = pd.read_csv(z.open(name), low_memory=False)
    df.columns = [str(c).split(":")[0].strip() for c in df.columns]
    for c in df.columns:
        if not pd.api.types.is_numeric_dtype(df[c]) and c != "TIME_PERIOD":      # object (pandas 2) and string (pandas 3) columns
            if c in ("UNIT_TYPE", "UNIT_MEASURE"):
                df[c + "_LABEL"] = df[c].astype(str)
            df[c] = df[c].astype(str).str.split(":").str[0].str.strip()
    return df


def bis_annual_last(d, keys, value="OBS_VALUE"):
    """Reduce a sub-annual BIS frame to one value per year (last period) per key."""
    d = d.copy()
    d["year"] = pd.to_numeric(d["TIME_PERIOD"].astype(str).str[:4], errors="coerce")
    d[value] = pd.to_numeric(d[value], errors="coerce")
    d = d.dropna(subset=["year", value]).sort_values("TIME_PERIOD")
    return d.groupby(keys + ["year"], as_index=False)[value].last()


def parse_bis_tc(df):
    cty = "BORROWERS_CTY" if "BORROWERS_CTY" in df.columns else "REF_AREA"
    d = df.copy()
    if "UNIT_TYPE_LABEL" in d.columns:
        d = d[d["UNIT_TYPE_LABEL"].str.contains("GDP", case=False)]
    if "TC_LENDERS" in d.columns:
        d = d[d["TC_LENDERS"] == "A"]
    if "TC_ADJUST" in d.columns and (d["TC_ADJUST"] == "A").any():
        d = d[d["TC_ADJUST"] == "A"]
    if "VALUATION" in d.columns:                                   # market value first, nominal as fallback
        d["_rk"] = d["VALUATION"].map({"M": 0, "N": 1}).fillna(2)
        d = d.sort_values("_rk").drop_duplicates([cty, "TC_BORROWERS", "TIME_PERIOD"])
    d = bis_annual_last(d, [cty, "TC_BORROWERS"])
    d["country_code"] = to_iso3(d[cty], src="ISO2").values
    d["indicator_code"] = "BIS.TC." + d["TC_BORROWERS"].astype(str) + ".PCT_GDP"          # G gov | H households | C NFC | P private NFS | N total NFS
    return tidy(d.rename(columns={"OBS_VALUE": "value"})[["country_code", "indicator_code", "year", "value"]], "BIS")


def parse_bis_cbpol(df):
    d = df.copy()
    if "FREQ" in d.columns and (d["FREQ"] == "M").any():
        d = d[d["FREQ"] == "M"]
    d["OBS_VALUE"] = pd.to_numeric(d["OBS_VALUE"], errors="coerce")
    d["year"] = pd.to_numeric(d["TIME_PERIOD"].astype(str).str[:4], errors="coerce")
    d = d.dropna(subset=["year", "OBS_VALUE"]).sort_values("TIME_PERIOD")
    g = d.groupby(["REF_AREA", "year"])["OBS_VALUE"]
    out = pd.concat([g.last().rename("END"), g.mean().rename("AVG")], axis=1).reset_index()
    rows = []
    for col in ("END", "AVG"):
        rows.append(pd.DataFrame({"country_code": to_iso3(out["REF_AREA"], src="ISO2").values, "year": out["year"],
                                  "indicator_code": f"BIS.CBPOL.POLICY_RATE_{col}", "value": out[col]}))
    return tidy(pd.concat(rows, ignore_index=True), "BIS")


bis_frames = []
for stem, parser in (("WS_TC", parse_bis_tc), ("WS_CBPOL", parse_bis_cbpol)):
    try:
        raw = bis_download(stem)
        if raw is None:
            print(f"  {stem}: download failed"); continue
        print(f"  {stem}: {len(raw)} rows, columns {list(raw.columns)[:14]}")
        part = parser(raw)
        part = part[part["country_code"].isin(set(target_countries) | {"EMU"})]
        bis_frames.append(part)
        print(f"     -> {len(part)} rows, {part['indicator_code'].nunique()} series, {part['country_code'].nunique()} countries")
    except Exception as e:
        print(f"  {stem} failed: {str(e)[:150]}")
for stem in BIS_INSPECT:
    raw = bis_download(stem)
    if raw is not None:
        print(f"--- {stem}: {len(raw)} rows")
        for c in raw.columns[:25]:
            print(f"   {c:<24} {raw[c].nunique():>6} distinct  e.g. {list(raw[c].dropna().astype(str).unique()[:6])}")
df_bis_final = pd.concat(bis_frames, ignore_index=True) if bis_frames else pd.DataFrame(columns=STD_COLS)
print(f"BIS: {len(df_bis_final)} rows, {df_bis_final['indicator_code'].nunique()} series")

In [ ]:
# 10. World Happiness Report (Gallup World Poll country-year panel)
WHR_URLS = [
    "https://files.worldhappiness.report/WHR26_Data_Figure_2.1.xlsx",
    "https://files.worldhappiness.report/WHR25_Data_Figure_2.1v3.xlsx",
    "https://files.worldhappiness.report/WHR24_Data_Figure_2.1.xlsx",
    "https://raw.githubusercontent.com/owid/owid-datasets/master/datasets/World%20Happiness%20Report/World%20Happiness%20Report.csv"
]

WHR_COLS = {
    "LIFE_LADDER": r"life ladder|ladder score|life evaluation",
    "SOCIAL_SUPPORT": r"social support",
    "POSITIVE_AFFECT": r"positive affect",
    "NEGATIVE_AFFECT": r"negative affect",
    "FREEDOM": r"freedom",
    "GENEROSITY": r"generosity",
    "PERCEPTIONS_CORRUPTION": r"corruption",
    "HEALTHY_LIFE_EXPECTANCY": r"healthy life"
}

def load_whr_data():
    for url in WHR_URLS:
        try:
            r = http_get(url, timeout=40, tries=2)
            log_fetch("WHR", r.status_code)
            if r.status_code != 200:
                continue
            if url.endswith(".csv"):
                return pd.read_csv(io.StringIO(r.text))
            else:
                xl = pd.ExcelFile(io.BytesIO(r.content))
                for sheet in xl.sheet_names:
                    d = xl.parse(sheet)
                    low = [str(c).lower() for c in d.columns]
                    if any("country" in c for c in low) and any("year" in c for c in low):
                        return d
        except Exception:
            continue
    return None

def parse_whr_per_country(raw_df, countries):
    if raw_df is None or raw_df.empty:
        return pd.DataFrame(columns=STD_COLS)
    
    cols_map = {str(c).lower().strip(): c for c in raw_df.columns}
    name_col = next((cols_map[c] for c in cols_map if "country" in c), None)
    year_col = next((cols_map[c] for c in cols_map if c == "year"), None)
    
    if not name_col or not year_col:
        return pd.DataFrame(columns=STD_COLS)

    raw_df["country_code"] = to_iso3(raw_df[name_col])
    out_frames = []
    
    # Process per country slice
    for iso3 in countries:
        c_sub = raw_df[raw_df["country_code"] == iso3]
        if c_sub.empty:
            continue
        
        for code, pattern in WHR_COLS.items():
            matched_col = next((cols_map[c] for c in cols_map if re.search(pattern, c) 
                                and not re.search(r"explained by|dystopia|whisker|rank", c)), None)
            if matched_col and matched_col in c_sub.columns:
                part = pd.DataFrame({
                    "country_code": iso3,
                    "indicator_code": "WHR." + code,
                    "year": c_sub[year_col],
                    "value": c_sub[matched_col]
                })
                out_frames.append(part)
                
    return tidy(pd.concat(out_frames, ignore_index=True), "WHR") if out_frames else pd.DataFrame(columns=STD_COLS)

raw_whr = load_whr_data()
df_whr = parse_whr_per_country(raw_whr, target_countries)
print(f"WHR: Ingested {len(df_whr)} rows across {df_whr['country_code'].nunique()} countries and {df_whr['indicator_code'].nunique()} series.")

In [ ]:
# 11. UNHCR - displacement by country of origin (refugees + asylum seekers + IDPs) -> health index "displaced by conflict"
# Per-country requests. Rows are summed by year (works whether the API returns one row per asylum country or already aggregated).
# Countries that answer 200 with no rows are zero-filled (no displaced population originating there).
UNHCR_URL = "https://api.unhcr.org/population/v1/population/"
UNHCR_FROM, UNHCR_TO = 2015, 2026


def unhcr_country(iso3):
    items, page = [], 1
    while True:
        r = http_get(UNHCR_URL, params={"coo": iso3, "coa_all": "true", "cf_type": "ISO", "yearFrom": UNHCR_FROM,
                                        "yearTo": UNHCR_TO, "limit": 1000, "page": page}, timeout=40)
        log_fetch("UNHCR", r.status_code)
        if r.status_code != 200:
            return None
        js = r.json()
        items += js.get("items", [])
        if page >= int(js.get("maxPages", 1) or 1):
            return items
        page += 1


def parse_unhcr(items, iso3):
    d = pd.DataFrame(items)
    if d.empty:
        return pd.DataFrame(columns=STD_COLS)
    fields = {"refugees": "UNHCR.REFUGEES_FROM", "asylum_seekers": "UNHCR.ASYLUM_SEEKERS_FROM", "idps": "UNHCR.IDPS"}
    for f in fields:
        d[f] = pd.to_numeric(d.get(f), errors="coerce").fillna(0)
    d["year"] = pd.to_numeric(d["year"], errors="coerce")
    g = d.groupby("year")[list(fields)].sum().reset_index()
    g["total"] = g[list(fields)].sum(axis=1)
    rows = [pd.DataFrame({"country_code": iso3, "indicator_code": code, "year": g["year"], "value": g[f]}) for f, code in fields.items()]
    rows.append(pd.DataFrame({"country_code": iso3, "indicator_code": "UNHCR.DISPLACED_TOTAL", "year": g["year"], "value": g["total"]}))
    return tidy(pd.concat(rows, ignore_index=True), "UNHCR")


unhcr_frames, unhcr_empty = [], []
for i, iso3 in enumerate(target_countries, 1):
    try:
        items = unhcr_country(iso3)
        if items is None:
            continue
        if not items:
            unhcr_empty.append(iso3)
        else:
            unhcr_frames.append(parse_unhcr(items, iso3))
    except Exception:
        log_fetch("UNHCR", "err")
    time.sleep(0.05)
    if i % 25 == 0 or i == len(target_countries):
        print(f"  [{i}/{len(target_countries)}] Processed {iso3}")
df_unhcr = pd.concat(unhcr_frames, ignore_index=True) if unhcr_frames else pd.DataFrame(columns=STD_COLS)
if not df_unhcr.empty and unhcr_empty:
    yrs = sorted(df_unhcr["year"].unique())
    zeros = [(c, code, y, 0.0, "UNHCR-derived") for c in unhcr_empty
             for code in ("UNHCR.DISPLACED_TOTAL", "UNHCR.REFUGEES_FROM", "UNHCR.ASYLUM_SEEKERS_FROM", "UNHCR.IDPS") for y in yrs]
    df_unhcr = pd.concat([df_unhcr, pd.DataFrame(zeros, columns=STD_COLS)], ignore_index=True)
print(f"UNHCR: {len(df_unhcr)} rows, {df_unhcr['country_code'].nunique()} countries ({len(unhcr_empty)} zero-filled)")

In [ ]:
# 11b. World Bank PIP - Gini and decile shares, one request per country
PIP_URL = "https://api.worldbank.org/pip/v1/pip"


def pip_country(iso3):
    r = http_get(PIP_URL, params={"country": iso3, "year": "all", "fill_gaps": "true", "format": "csv"}, timeout=60)
    log_fetch("PIP", r.status_code)
    if r.status_code != 200 or not r.text.strip():
        return None
    return pd.read_csv(io.StringIO(r.text))


def _pct(s):
    s = pd.to_numeric(s, errors="coerce")
    return s * 100 if s.dropna().le(1.0).all() else s          # PIP returns fractions; WDI uses 0-100


def parse_pip(d, iso3):
    d = d.copy()
    d.columns = [str(c).lower() for c in d.columns]
    ycol = next((c for c in ("reporting_year", "year") if c in d.columns), None)
    if ycol is None or "gini" not in d.columns:
        return pd.DataFrame(columns=STD_COLS)
    if "reporting_level" in d.columns:                             # national rows when available (China / India / Indonesia also have urban / rural)
        nat = d[d["reporting_level"].astype(str).str.lower() == "national"]
        d = nat if len(nat) else d
    if "welfare_type" in d.columns:                                # one row per year: consumption before income
        d = d.sort_values("welfare_type", kind="stable")
    d = d.drop_duplicates(ycol)
    series = {"PIP.GINI": _pct(d["gini"]), "PIP.DECILE1": _pct(d["decile1"]) if "decile1" in d.columns else None,
              "PIP.DECILE10": _pct(d["decile10"]) if "decile10" in d.columns else None,
              "PIP.MLD": pd.to_numeric(d["mld"], errors="coerce") if "mld" in d.columns else None}
    rows = [pd.DataFrame({"country_code": iso3, "indicator_code": code, "year": d[ycol].values, "value": s.values})
            for code, s in series.items() if s is not None]
    out = tidy(pd.concat(rows, ignore_index=True), "PIP")
    return out[out["year"] >= 2000]


pip_frames, pip_none = [], []
for i, iso3 in enumerate(target_countries, 1):
    try:
        d = pip_country(iso3)
        if d is None or d.empty:
            pip_none.append(iso3)
        else:
            if not pip_frames:
                print("  PIP response columns:", list(d.columns)[:30])
            part = parse_pip(d, iso3)
            (pip_frames if not part.empty else pip_none).append(part if not part.empty else iso3)
    except Exception as e:
        log_fetch("PIP", "err")
        pip_none.append(iso3)
    time.sleep(0.1)
    if i % 25 == 0 or i == len(target_countries):
        print(f"  [{i}/{len(target_countries)}] Processed {iso3}")
df_pip = pd.concat(pip_frames, ignore_index=True) if pip_frames else pd.DataFrame(columns=STD_COLS)
print(f"PIP: {len(df_pip)} rows, {df_pip['country_code'].nunique()} countries, {df_pip['indicator_code'].nunique()} series; "
      f"no data for {len(pip_none)} countries")

In [ ]:
# 12. OECD - National Accounts at a Glance, chapter 5 "Households" (household saving rate, income, debt ...)
# OECD members + a few partners only (the OECD API allows ~60 calls/hour). Key: FREQ.REF_AREA.MEASURE.UNIT_MEASURE.CHAPTER, wildcards omitted.
OECD_BASE = "https://sdmx.oecd.org/public/rest/data/"
OECD_FLOWS = ["OECD.SDD.NAD,DSD_NAAG@DF_NAAG_V,1.0", "OECD.SDD.NAD,DSD_NAAG@DF_NAAG_V"]
OECD_COUNTRIES = ["AUS", "AUT", "BEL", "CAN", "CHL", "COL", "CRI", "CZE", "DNK", "EST", "FIN", "FRA", "DEU", "GRC", "HUN", "ISL",
                  "IRL", "ISR", "ITA", "JPN", "KOR", "LVA", "LTU", "LUX", "MEX", "NLD", "NZL", "NOR", "POL", "PRT", "SVK", "SVN",
                  "ESP", "SWE", "CHE", "TUR", "GBR", "USA", "BRA", "CHN", "IND", "IDN", "ZAF", "ARG", "RUS", "SAU"]


def parse_oecd_csv(text, iso3):
    d = pd.read_csv(io.StringIO(text))
    need = {"MEASURE", "UNIT_MEASURE", "TIME_PERIOD", "OBS_VALUE"}
    if d.empty or not need <= set(d.columns):
        return pd.DataFrame(columns=STD_COLS)
    d["indicator_code"] = "OECD.NAAG_V." + d["MEASURE"].astype(str) + "." + d["UNIT_MEASURE"].astype(str)
    d["country_code"], d["year"], d["value"] = iso3, d["TIME_PERIOD"], d["OBS_VALUE"]
    out = tidy(d[["country_code", "indicator_code", "year", "value"]], "OECD")
    alias = out[out["indicator_code"] == "OECD.NAAG_V.B8NS1M.PT_B6N_S1M"].assign(indicator_code="OECD.HH_SAVINGS_RATE")   # net household saving rate
    return pd.concat([out, alias], ignore_index=True)


oecd_frames, oecd_flow = [], None
for iso3 in [c for c in OECD_COUNTRIES if c in set(target_countries)]:
    for flow in ([oecd_flow] if oecd_flow else OECD_FLOWS):
        try:
            r = http_get(f"{OECD_BASE}{flow}/A.{iso3}...", params={"startPeriod": "2000", "format": "csvfile"}, timeout=60, tries=2)
            log_fetch("OECD", r.status_code)
            if r.status_code == 200 and r.text.strip():
                part = parse_oecd_csv(r.text, iso3)
                if not part.empty:
                    oecd_frames.append(part)
                    oecd_flow = flow
                    break
        except Exception:
            log_fetch("OECD", "err")
    time.sleep(1.0)
df_oecd = pd.concat(oecd_frames, ignore_index=True) if oecd_frames else pd.DataFrame(columns=STD_COLS)
print(f"OECD NAAG_V: {len(df_oecd)} rows, {df_oecd['country_code'].nunique()} countries, {df_oecd['indicator_code'].nunique()} series (flow used: {oecd_flow})")

In [ ]:
# 12b. ARWU (Shanghai Ranking) -> universities in top 100 / top 500 per country
ARWU_API = "https://www.shanghairanking.com/api/pub/v1/arwu/rank"
ARWU_YEARS = range(2019, 2027)
ARWU_REGION_ALIAS = {"Chinese Mainland": "CHN", "Mainland China": "CHN", "Taiwan Region": "TWN", "Hong Kong SAR": "HKG",
                     "Macau SAR": "MAC", "Korea": "KOR", "Russia": "RUS", "UK": "GBR", "US": "USA"}


def arwu_rows(js):
    data = js.get("data", js) if isinstance(js, dict) else js
    if isinstance(data, dict):
        for k in ("rankings", "rank", "list", "univs"):
            if isinstance(data.get(k), list):
                return data[k]
        for v in data.values():
            if isinstance(v, list) and v and isinstance(v[0], dict):
                return v
        return []
    return data if isinstance(data, list) else []


def _first_key(row, names):
    return next((n for n in names if n in row), None)


def arwu_counts(rows, year, countries):
    if not rows:
        return pd.DataFrame(columns=STD_COLS)
    rk_key = _first_key(rows[0], ("ranking", "rank", "rankingNo", "rankNo"))
    rg_key = _first_key(rows[0], ("region", "regionName", "country", "countryName"))
    if rk_key is None or rg_key is None:
        print(f"  ARWU {year}: unknown row layout, keys = {list(rows[0])[:15]}")
        return pd.DataFrame(columns=STD_COLS)
    d = pd.DataFrame({"rank": [(lambda m: int(m.group(1)) if m else np.nan)(re.match(r"\s*(\d+)", str(r.get(rk_key, "")))) for r in rows],
                      "region": [r.get(rg_key) for r in rows]}).dropna()
    alias = d["region"].map(ARWU_REGION_ALIAS)
    d["iso3"] = alias.where(alias.notna(), to_iso3(d["region"]).values)
    d = d.dropna(subset=["iso3"])
    out = []
    for code, cut in (("ARWU.TOP100_COUNT", 100), ("ARWU.TOP500_COUNT", 500)):
        cnt = d[d["rank"] <= cut].groupby("iso3").size()
        for c in set(countries) | set(cnt.index):
            out.append((c, code, year, float(cnt.get(c, 0)), "ARWU"))
    return tidy(pd.DataFrame(out, columns=STD_COLS))


arwu_frames = []
for y in ARWU_YEARS:
    try:
        r = http_get(ARWU_API, params={"version": y}, timeout=40, tries=2)
        log_fetch("ARWU", r.status_code)
        if r.status_code != 200:
            print(f"  ARWU {y}: HTTP {r.status_code}")
            continue
        rows = arwu_rows(r.json())
        if rows and not arwu_frames:
            print("  ARWU first row keys:", list(rows[0])[:15])
        part = arwu_counts(rows, y, target_countries)
        if not part.empty:
            arwu_frames.append(part)
    except Exception as e:
        print(f"  ARWU {y} failed: {str(e)[:120]}")
    time.sleep(0.3)
df_arwu = pd.concat(arwu_frames, ignore_index=True) if arwu_frames else pd.DataFrame(columns=STD_COLS)
print(f"ARWU: {len(df_arwu)} rows, years {sorted(df_arwu['year'].unique()) if len(df_arwu) else []}")

In [ ]:
# 13. Penn World Table 11.0 (Groningen)
PWT_DOI = "doi:10.34894/FABVLR"
PWT_VARS = ["rgdpo", "rgdpna", "pop", "emp", "avh", "hc", "ctfp", "rtfpna", "labsh", "rkna", "rnna", "delta", "csh_i", "irr"]
PWT_MIN_YEAR = 1995

def parse_pwt(df, countries):
    d = df[df["year"] >= PWT_MIN_YEAR]
    by_c = {k: g for k, g in d.groupby("countrycode")}
    rows = []
    for iso3 in countries:                              # per-country slice
        g = by_c.get(iso3)
        if g is None:
            continue
        have = [v for v in PWT_VARS if v in g.columns]
        m = pd.melt(g, id_vars=["countrycode", "year"], value_vars=have, var_name="indicator_code", value_name="value")
        m["indicator_code"] = "PWT." + m["indicator_code"].str.upper()
        rows.append(m.rename(columns={"countrycode": "country_code"}))
    out = tidy(pd.concat(rows, ignore_index=True), "PWT") if rows else pd.DataFrame(columns=STD_COLS)
    extra = []
    for iso3, g in out.groupby("country_code"):
        w = g.pivot_table(index="year", columns="indicator_code", values="value", aggfunc="first").sort_index()
        if {"PWT.RGDPO", "PWT.EMP"} <= set(w.columns):
            s = (w["PWT.RGDPO"] / w["PWT.EMP"] * 1e6).dropna()                   # US$ per worker (rgdpo in mil. 2021 US$, emp in millions)
            extra.append(pd.DataFrame({"country_code": iso3, "indicator_code": "PWT.RGDPO_PER_WORKER", "year": s.index, "value": s.values}))
        if "PWT.RKNA" in w.columns:
            s = (w["PWT.RKNA"].pct_change(fill_method=None) * 100).dropna()
            extra.append(pd.DataFrame({"country_code": iso3, "indicator_code": "PWT.RKNA_GROWTH", "year": s.index, "value": s.values}))
    return pd.concat([out] + [tidy(e, "PWT") for e in extra], ignore_index=True) if extra else out

def pwt_load_native():
    meta = http_get("https://dataverse.nl/api/datasets/:persistentId/", params={"persistentId": PWT_DOI}, timeout=40).json()
    files = meta["data"]["latestVersion"]["files"]
    cands = []
    for f in files:
        fn = f.get("dataFile", {}).get("filename", f.get("label", ""))
        m = re.fullmatch(r"(?i)pwt\d+\.(xlsx|dta)", fn)
        if m:
            # Prioritize Stata (.dta) because pd.read_stata has zero third-party C/Python dependencies
            cands.append((0 if fn.lower().endswith("dta") else 1, fn, f["dataFile"]["id"]))
    if not cands:
        return None
    _, fn, fid = sorted(cands)[0]
    r = http_get(f"https://dataverse.nl/api/access/datafile/{fid}", params={"format": "original"}, timeout=300, tries=2)
    log_fetch("PWT", r.status_code)
    r.raise_for_status()
    print("  PWT loaded file format:", fn)
    if fn.lower().endswith("dta"):
        return pd.read_stata(io.BytesIO(r.content))
    return pd.read_excel(io.BytesIO(r.content), sheet_name="Data")

try:
    pwt_raw = pwt_load_native()
    df_pwt = parse_pwt(pwt_raw, target_countries) if pwt_raw is not None else pd.DataFrame(columns=STD_COLS)
except Exception as e:
    print(f"PWT failed: {e}")
    df_pwt = pd.DataFrame(columns=STD_COLS)

print(f"PWT: {len(df_pwt)} rows, {df_pwt['indicator_code'].nunique()} series across {df_pwt['country_code'].nunique()} countries.")

In [ ]:
# 14. Gauge trees with indicators attached
# Leaf convention:  (components, weight)
#   components = list; every entry is ONE component of the leaf (averaged after standardisation, missing ones re-normalised)
#   each entry is alt("A", "B", ...) = alternatives for the same component: first available per country wins (coalesce)
#   a code ending in "*" matches every series with that prefix
#   codes marked  # PLANNED  are not fetched yet; the checker reports them as missing, so the tree doubles as a to-do list
#   direction (+/-), transforms (log, per-capita, relative-to-similar-income) are NOT modelled here yet
# Strength leaves: ({"total": (components, w), "per_capita": (components, w)}, weight);  "@..." entries are references to other nodes.


def alt(*codes):
    return tuple(codes)


HOURS = alt("ILO.HOURS_WEEKLY_MODELLED", "PWT.HOURS_PER_WORKER", "PWT.AVH")
GDP_PC_COST = alt("IMF.NGDPDPC", "NY.GDP.PCAP.CD")                       # report p.160-161: nominal GDP per capita is the cost proxy
PISA_HCI = alt("LO.PISA.MAT", "HD.HCI.HLOS", "LO.PISA.REA", "LO.PISA.SCI")

FUTURE_GROWTH_FORMULA = {
    "type": "additive_growth_identity",
    "formula": "working_age_pop_growth + growth_per_worker_estimate + error_adj",
    "working_age_pop_growth": [alt("DERIVED.WORKING_AGE_GROWTH_FWD10")],
    "growth_per_worker_estimate": {
        "type": "weighted_growth_impact",
        "components": {"productivity_implied_growth": 0.65, "indebtedness_implied_growth": 0.35},
    },
    # error_adj needs a backtest history of the estimate (report p.147): use the report's published adjusted/unadjusted estimates (pp.146-152) as targets.
}

RAY_DALIO_GAUGE_TREES = {
    "productivity_gauge": (
        {
            "what_you_pay_vs_what_you_get": (
                {
                    "education_cost_quality_adj": ([
                        alt("WAGE.EDUCATED_HOURLY_USD", *GDP_PC_COST),                      # PLANNED wage data -> GDP-per-capita proxy
                        PISA_HCI,
                        alt("BAR.SCHL.25UP", "HD.HCI.EYRS", "SE.TER.CUAT.BA.ZS"),
                        HOURS,
                    ], 0.25),
                    "labor_productivity_cost_adj": ([
                        GDP_PC_COST,
                        alt("PWT.RGDPO_PER_WORKER", "SL.GDP.PCAP.EM.KD"),
                        HOURS,
                    ], 0.25),
                    "working_hard": (
                        {
                            "avg_hours_worked_per_week": ([HOURS], 2 / 3),
                            "dependency_ratio_10y_fwd_change": ([alt("DERIVED.DEPENDENCY_RATIO_CHG_FWD10")], 1 / 3),
                        },
                        0.25,
                    ),
                    "investing": (
                        {
                            "investment_pct_gdp": ([alt("NE.GDI.FTOT.ZS", "IMF.NID_NGDP", "NE.GDI.TOTL.ZS")], 3.1 / 17.5),
                            "household_savings_rate": ([alt("OECD.HH_SAVINGS_RATE", "IMF.NGSD_NGDP", "NY.GNS.ICTR.ZS", "NY.GDS.TOTL.ZS")], 3.1 / 17.5),
                            "growth_in_capital_stock": ([alt("PWT.RKNA_GROWTH", "DERIVED.CAPITAL_STOCK_GROWTH_PIM")], 3.1 / 17.5),
                            "productivity_of_capital": ([alt("DERIVED.CAPITAL_PRODUCTIVITY")], 3.1 / 17.5),
                            "income_volatility": ([alt("DERIVED.GDP_GROWTH_VOL10")], 2.6 / 17.5),
                            "inflation_volatility": ([alt("DERIVED.INFLATION_VOL10")], 2.6 / 17.5),
                        },
                        0.25,
                    ),
                },
                0.70,
            ),
            "culture": (
                {
                    "corruption": (
                        {
                            "transparency_index": ([alt("TI.CPI", "CC.PER.RNK", "CC.EST")], 0.50),
                            "surveys_of_corruption": (
                                {
                                    "diversion_of_public_funds": ([alt("VDEM.v2exthftps", "VDEM.v2x_pubcorr")], 1 / 3),
                                    "irregular_payments_and_bribes": ([alt("VDEM.v2excrptps", "VDEM.v2x_execorr")], 1 / 3),
                                    "favoritism_in_govt_decisions": ([alt("VDEM.v2clrspct", "GE.PER.RNK", "GE.EST")], 1 / 3),
                                },
                                0.50,
                            ),
                        },
                        1 / 6,
                    ),
                    "bureaucracy": (
                        {
                            "starting_a_business": ([alt("IC.REG.STRT.BUS.DFRN", "IC.REG.DFRN.PC.DFRN")], 1 / 3),      # DB archive, ends 2019/20
                            "construction_permits": ([alt("IC.CNST.PRMT.DFRN.DB1619", "IC.CNST.PRMT.DFRN.DB0615")], 1 / 3),
                            "burden_of_govt_regulation": ([alt("RQ.PER.RNK", "RQ.EST")], 1 / 3),
                        },
                        1 / 6,
                    ),
                    "rule_of_law": (
                        {
                            "efficiency_of_legal_frameworks": ([alt("VDEM.v2cltrnslw", "VDEM.v2x_jucon", "VDEM.v2xcl_rol")], 0.25),
                            "property_rights": ([alt("VDEM.v2xcl_prpty", "RL.PER.RNK", "RL.EST")], 0.25),
                            "protecting_investors_value": ([alt("PROT.MINOR.INV.DFRN.DB1519", "PROT.MINOR.INV.DFRN.DB0614")], 0.25),
                            "enforcing_contracts": ([alt("ENF.CONT.COEN.DB1719.DFRN", "ENF.CONT.COEN.DB0415.DFRN")], 0.25),
                        },
                        1 / 6,
                    ),
                    "savoring_life_vs_achieving": (
                        {
                            "observed_outcomes_hours_worked": ([HOURS], 0.50),
                            "expressed_values_surveys": ([alt("WVS.ACHIEVEMENT_VALUES")], 0.50),                       # PLANNED (World Values Survey)
                        },
                        1 / 6,
                    ),
                    "innovation_commercialism": (
                        {
                            "innovation_inputs": ([
                                alt("GB.XPD.RSDV.GD.ZS"), alt("SP.POP.SCIE.RD.P6"),
                                alt("GEM.FEAR_OF_FAILURE"), alt("GEM.ENTREPRENEURSHIP_PREVALENCE"),                     # PLANNED (GEM)
                            ], 0.50),
                            "innovation_outputs": ([
                                alt("IP.PAT.RESD", "IP.PAT.NRES"), alt("BX.GSR.ROYL.CD"), alt("IC.BUS.NREG"),
                                alt("IC.BUS.NDNS.ZS"), alt("IP.TMK.RSCT"), alt("WIPO.NEW_MAJOR_WEBSITES"),              # last PLANNED
                            ], 0.50),
                        },
                        1 / 6,
                    ),
                    "self_sufficiency": (
                        {
                            "hard_work": ([
                                HOURS, alt("SL.TLF.CACT.ZS", "ILO.LFPR_MODELLED"),
                                alt("OECD.EFFECTIVE_RETIREMENT_AGE"), alt("VACATION.DAYS"),                            # PLANNED
                            ], 0.50),
                            "government_support": ([alt("GC.XPN.TRFT.ZS"), alt("GC.XPN.TOTL.GD.ZS", "IMF.GGX_NGDP")], 0.25),
                            "labor_market_rigidity": ([
                                alt("ILO.COLLECTIVE_BARGAINING_COVERAGE*"), alt("OECD.EPL", "WEF.HIRING_FIRING"),       # EPL/WEF PLANNED
                                alt("ILO.MIN_WAGE*"),
                            ], 0.25),
                        },
                        1 / 6,
                    ),
                },
                0.30,
            ),
        },
        1.0,
    ),

    "indebtedness_gauge": (
        {
            "debt_levels_and_flow": ([
                alt("BIS.TC.G.PCT_GDP", "IMF.GGXWDG_NGDP", "GC.DOD.TOTL.GD.ZS"),                                     # government debt
                alt("BIS.TC.P.PCT_GDP", "BIS.TC.N.PCT_GDP", "FS.AST.PRVT.GD.ZS"),                                    # private / total non-financial debt
                alt("DERIVED.DEBT_GROWTH_MINUS_INCOME_GROWTH"),
                alt("IMF.GGXCNL_NGDP", "GC.NLD.TOTL.GD.ZS"),
                alt("IMF.BCA_NGDPD", "BN.CAB.XOKA.GD.ZS"),
                alt("GC.XPN.INTP.RV.ZS", "GC.XPN.INTP.ZS"),
                alt("DT.DOD.DECT.GN.ZS", "DT.TDS.DECT.EX.ZS"),                                                       # external / FX-denominated debt proxy (LMICs only)
                alt("FI.RES.TOTL.MO"),
                alt("RATING.SOVEREIGN"), alt("DEBT.FOREIGN_HELD_SHARE"),                                             # PLANNED, no free API identified
            ], 0.50),
            "monetary_policy": ([
                alt("BIS.CBPOL.POLICY_RATE_END", "BIS.CBPOL.POLICY_RATE_AVG"),
                alt("FR.INR.RINR"), alt("FR.INR.LEND"), alt("FM.LBL.BMNY.ZG"),
                alt("IMF.PCPIPCH", "FP.CPI.TOTL.ZG"),
            ], 0.50),
        },
        1.0,
    ),

    "health_index": (
        {
            "life_expectancy_at_birth": ([alt("SP.DYN.LE00.IN")], 0.300),
            "share_facing_food_insecurity": ([alt("SN.ITK.MSFI.ZS", "SN.ITK.DEFC.ZS")], 0.150),
            "access_to_essential_health_services": ([alt("SH.UHC.SRVS.CV.XD", "SH.XPD.CHEX.GD.ZS", "SH.MED.PHYS.ZS")], 0.150),
            "child_mortality_rate": ([alt("SH.DYN.MORT", "SP.DYN.IMRT.IN")], 0.100),
            "homicide_rate": ([alt("VC.IHR.PSRC.P5")], 0.075),
            "displaced_by_conflict": ([alt("UNHCR.DISPLACED_PCT_POP")], 0.075),
            "air_pollution_exposure_ppm": ([alt("EN.ATM.PM25.MC.M3")], 0.050),
            "safely_managed_drinking_water": ([alt("SH.H2O.SMDW.ZS", "SH.H2O.BASW.ZS")], 0.050),
            "basic_sanitation_facilities": ([alt("SH.STA.BASS.ZS", "SH.STA.SMSS.ZS")], 0.050),
        },
        1.0,
    ),
    "happiness_index": (
        {
            "share_reported_happy": ([alt("WHR.POSITIVE_AFFECT")], 0.225),                       # Gallup "happy"/enjoyment composite is not published separately
            "share_reported_daily_enjoyment": ([alt("WHR.POSITIVE_AFFECT")], 0.225),
            "reported_life_satisfaction": ([alt("WHR.LIFE_LADDER", "WHR.CANTRIL_LADDER")], 0.225),
            "good_social_support_network": ([alt("WHR.SOCIAL_SUPPORT")], 0.225),
            "suicide_rate_per_100k": ([alt("SH.STA.SUIC.P5")], 0.100),
        },
        1.0,
    ),

    "internal_order_cycle": (
        {
            "wealth_values_gap": ([
                alt("SI.POV.GINI"), alt("SI.DST.10TH.10", "WID.TOP10_INCOME"), alt("WID.TOP1_WEALTH"),                  # WID PLANNED
                alt("VDEM.v2xeg_eqdr", "VDEM.v2x_egaldem"), alt("VDEM.v2cacamps", "VDEM.v2smpolsoc"),
            ], 0.50),
            "internal_conflict": ([
                alt("VDEM.v2cagenmob", "VDEM.v2cademmob"), alt("VDEM.v2x_clphy"), alt("PV.PER.RNK", "PV.EST"),
                alt("GDELT.PROTEST_INTENSITY", "ACLED.PROTESTS"), alt("UCDP.INTERNAL_CONFLICT_DEATHS"),                  # PLANNED
            ], 0.50),
        },
        1.0,
    ),
    "economic_financial_cycle": (
        {
            "debt_burden": (["@indebtedness_gauge"], 0.50),
            "expected_growth": (["@future_growth"], 0.50),
        },
        1.0,
    ),
    "external_order_cycle": ([                                                                                       # pairwise data: mostly PLANNED
        alt("TM.TAX.MRCH.WM.AR.ZS"), alt("MS.MIL.XPND.GD.ZS"),
        alt("DOTS.BILATERAL_TRADE_CONCENTRATION"), alt("UN.VOTING_DISTANCE"), alt("SANCTIONS.GSDB"), alt("UCDP.INTERSTATE"),
    ], 1.0),

    "overall_strength_gauge_heuristic": (
        {
            "key_eight_measures_of_power": (
                {
                    "economic_output": ({
                        "total": ([alt("IMF.PPPGDP", "NY.GDP.MKTP.PP.KD"), alt("IMF.NGDPD", "NY.GDP.MKTP.CD")], 0.5),
                        "per_capita": ([alt("IMF.PPPPC", "NY.GDP.PCAP.PP.KD"), alt("IMF.NGDPDPC", "NY.GDP.PCAP.CD")], 0.5),
                    }, 0.15),
                    "financial_center": ({
                        "total": ([alt("CM.MKT.LCAP.CD"), alt("CM.MKT.TRAD.CD"), alt("BIS.DEBT_SEC.OUTSTANDING"),
                                   alt("BIS.LBS.CROSSBORDER_CLAIMS"), alt("BIS.FX_TURNOVER_VALUE")], 0.5),              # BIS items PLANNED (BIS_INSPECT)
                        "per_capita": ([alt("CM.MKT.LCAP.GD.ZS"), alt("CM.MKT.TRNR"), alt("FS.AST.PRVT.GD.ZS"),
                                        alt("GFCI.SCORE")], 0.5),                                                       # GFCI PLANNED
                    }, 0.15),
                    "reserve_currency_status": ({
                        "total": ([alt("IMF.COFER_SHARE"), alt("BIS.FX_TURNOVER_SHARE"), alt("BIS.INTL_DEBT_CCY_SHARE")], 1.0),   # last two PLANNED
                    }, 0.12),
                    "military_strength": ({
                        "total": ([alt("MS.MIL.XPND.CD"), alt("MS.MIL.TOTL.P1"), alt("FAS.NUCLEAR_STOCKPILE"), alt("MS.MIL.XPRT.KD")], 0.5),
                        "per_capita": ([alt("MS.MIL.XPND.GD.ZS"), alt("MS.MIL.XPND.ZS"), alt("MS.MIL.TOTL.P1")], 0.5),        # personnel / population
                    }, 0.12),
                    "innovation_and_technology": ({
                        "total": ([alt("IP.PAT.RESD", "IP.PAT.NRES"), alt("GB.XPD.RSDV.GD.ZS"), alt("SP.POP.SCIE.RD.P6"),
                                   alt("IP.JRN.ARTC.SC"), alt("BX.GSR.ROYL.CD"), alt("CSET.AI_OUTPUT")], 0.5),             # last PLANNED
                        "per_capita": ([alt("GB.XPD.RSDV.GD.ZS"), alt("SP.POP.SCIE.RD.P6"), alt("TX.MNF.TECH.ZS.UN", "TX.VAL.TECH.MF.ZS"),
                                        alt("IT.NET.SECR.P6"), alt("WIPO.GII_SCORE")], 0.5),                               # GII PLANNED
                    }, 0.12),
                    "trade": ({
                        "total": ([alt("BX.GSR.GNFS.CD", "NE.EXP.GNFS.CD"), alt("TX.VAL.MRCH.CD.WT")], 0.5),
                        "per_capita": ([alt("NE.TRD.GNFS.ZS"), alt("TX.VAL.TECH.MF.ZS", "TX.MNF.TECH.ZS.UN")], 0.5),
                    }, 0.10),
                    "education": ({
                        "total": ([alt("SE.TER.ENRR"), alt("SE.SEC.ENRL"), alt("SE.PRM.ENRL"), alt("ARWU.TOP500_COUNT")], 0.5),   # ARWU PLANNED
                        "per_capita": ([PISA_HCI, alt("BAR.SCHL.25UP", "HD.HCI.EYRS"), alt("SE.SEC.CUAT.UP.ZS"),
                                        alt("SE.XPD.TOTL.GD.ZS")], 0.5),
                    }, 0.10),
                    "competitiveness": ({
                        "total": (["@productivity_gauge.what_you_pay_vs_what_you_get"], 1.0),
                    }, 0.08),
                },
                0.80,
            ),
            "additional_measures_of_power": (
                {
                    "infrastructure_and_investment": ([
                        alt("NE.GDI.FTOT.CD", "NE.GDI.TOTL.CD"), alt("NE.GDI.FTOT.ZS", "IMF.NID_NGDP"),
                        alt("NY.GNS.ICTR.ZS", "IMF.NGSD_NGDP"), alt("LP.LPI.OVRL.XQ"),
                        alt("OWID.ELECTRICITY_GEN_TWH", "EIA.ELECTRICITY*", "EG.USE.ELEC.KH.PC"), alt("EG.ELC.ACCS.ZS"),
                        alt("IS.AIR.PSGR"), alt("IS.AIR.GOOD.MT.K1"), alt("IS.RRS.TOTL.KM"), alt("IS.RRS.GOOD.MT.K6"),
                        alt("IS.SHP.GCNW.XQ"), alt("IT.NET.BBND.P2", "IT.NET.USER.ZS"),
                    ], 0.20),
                    "resource_allocation_efficiency": ([
                        alt("ILO.UNEMP_RATE_MODELLED", "SL.UEM.TOTL.ZS", "IMF.LUR"), alt("ILO.UNEMP_LONGTERM_SHARE*"),
                        alt("ILO.EMP_POP_RATIO_MODELLED", "SL.EMP.TOTL.SP.ZS"), alt("DERIVED.DEBT_GROWTH_MINUS_INCOME_GROWTH"),
                        alt("IC.CRED.ACC.LGL.RGHT.012.XD.DB1519.DFRN"), alt("OECD.EPL"),
                    ], 0.20),
                    "geology": ([
                        alt("AG.LND.TOTL.K2"), alt("EIA.ENERGY.PRODUCTION_QBTU", "OWID.PRIMARY_ENERGY_TWH"),
                        alt("OWID.OIL_PRODUCTION_TWH"), alt("OWID.GAS_PRODUCTION_TWH"), alt("OWID.COAL_PRODUCTION_TWH"),
                        alt("AG.PRD.CREL.MT", "AG.PRD.FOOD.XD"), alt("USGS.METALS_PRODUCTION", "TX.VAL.MMTL.ZS.UN"),
                        alt("EIA.ENERGY.SELF_SUFF", "EG.IMP.CONS.ZS"), alt("ER.H2O.INTR.K3", "ER.H2O.INTR.PC"),
                    ], 0.20),
                    "character_determination_civility": ([
                        HOURS, alt("SL.TLF.CACT.MA.ZS", "ILO.LFPR_MODELLED"), alt("GC.XPN.TRFT.ZS"),
                        alt("OECD.EFFECTIVE_RETIREMENT_AGE"), alt("WVS.ACHIEVEMENT_VALUES"), alt("VDEM.v2smpolhate", "VDEM.v2smpolsoc"),
                    ], 0.15),
                    "governance_rule_of_law": ([
                        alt("RL.PER.RNK", "RL.EST"), alt("CC.PER.RNK", "CC.EST"), alt("GE.PER.RNK", "GE.EST"),
                        alt("RQ.PER.RNK", "RQ.EST"), alt("VDEM.v2x_rule"), alt("VDEM.v2x_corr"),
                    ], 0.15),
                    "acts_of_nature": ([
                        alt("NDGAIN.GAIN"), alt("NDGAIN.VULNERABILITY"), alt("NDGAIN.READINESS"),
                        alt("COVID.EXCESS_MORTALITY_PSCORE"),
                    ], 0.10),
                },
                0.20,
            ),
        },
        1.0,
    ),
}

In [ ]:
# 14b. Tree utilities: weight check, code -> leaf lookup, leaf coverage
def _is_leaf(node):
    return isinstance(node[0], list)


def iter_leaves(node, path=(), w=1.0):
    """Yield (path, components, effective_weight). A node is (children_dict | components_list, weight)."""
    body, weight = node
    w = w * weight
    if _is_leaf(node):
        yield "/".join(path), body, w
    else:
        for name, child in body.items():
            yield from iter_leaves(child, path + (name,), w)


def check_weights(trees):
    bad = []

    def walk(node, path):
        body, _ = node
        if isinstance(body, dict):
            s = sum(ch[1] for ch in body.values())
            if abs(s - 1.0) > 1e-3:
                bad.append((path, round(s, 4)))
            for k, ch in body.items():
                walk(ch, f"{path}/{k}")
    for name, tree in trees.items():
        walk(tree, name)
    return bad       # known: investing sums to 1.0057 (report rounding), key_eight to 0.94 (draft heuristics)


def _code_matches(pattern, codes):
    return [c for c in codes if c.startswith(pattern[:-1])] if pattern.endswith("*") else ([pattern] if pattern in codes else [])


CODE_LEAVES = defaultdict(list)                       # exact code or "PREFIX*" -> [leaf paths]
for _t, _tree in RAY_DALIO_GAUGE_TREES.items():
    for _path, _comps, _w in iter_leaves(_tree, (_t,)):
        for _comp in _comps:
            if not isinstance(_comp, str):
                for _code in _comp:
                    CODE_LEAVES[_code].append(_path)


def resolve_gauge(code):
    """Top-level gauge (+ first leaf) a fetched code feeds; 'unassigned' if no leaf lists it."""
    leaves = CODE_LEAVES.get(code)
    if not leaves:
        for pat, lv in CODE_LEAVES.items():
            if pat.endswith("*") and code.startswith(pat[:-1]):
                leaves = lv
                break
    if not leaves:
        return "unassigned"
    return leaves[0].split("/")[0] + ": " + leaves[0].split("/")[-1] + (f" (+{len(leaves) - 1})" if len(leaves) > 1 else "")


def leaf_coverage(trees, df_master, universe, min_year=2019):
    """Per leaf: which components exist in df_master and how many universe countries the (coalesced) alternatives reach."""
    d = df_master[(df_master["year"] >= min_year) & (~df_master["is_projection"])]
    reach = d.groupby("indicator_code")["country_code"].agg(lambda s: set(s))
    codes, uni, rows = set(reach.index), set(universe), []
    for tname, tree in trees.items():
        for path, comps, w in iter_leaves(tree, (tname,)):
            n_comp, n_ok, cov, missing = 0, 0, [], []
            for comp in comps:
                if isinstance(comp, str):                      # '@reference'
                    continue
                n_comp += 1
                hit = [h for pat in comp for h in _code_matches(pat, codes)]
                if hit:
                    n_ok += 1
                    cov.append(len(set().union(*[reach[h] for h in hit]) & uni) / len(uni))
                else:
                    missing.append(comp[0])
            rows.append({"leaf": path, "eff_weight": round(w, 4), "components": n_comp, "available": n_ok,
                         "mean_cov_of_available_%": round(100 * float(np.mean(cov)), 1) if cov else 0.0,
                         "missing_first_choice": missing})
    out = pd.DataFrame(rows)
    out["weight_share_available"] = (out["available"] / out["components"].replace(0, np.nan)).fillna(1.0)
    return out


DALIO_24 = ["USA", "CHN", "DEU", "JPN", "KOR", "IND", "GBR", "FRA", "RUS", "SGP", "AUS", "TUR", "CAN", "CHE", "BRA",
            "NLD", "IDN", "ITA", "ESP", "SAU", "MEX", "ZAF", "ARG"]                      # + "EMU" (Eurozone aggregate) is the 24th entity
DALIO_35 = DALIO_24 + ["ARE", "POL", "COL", "THA", "CHL", "SWE", "HUN", "CZE", "PRT", "BEL", "NOR", "GRC"]

In [ ]:
# 14c. Tree patches
def _node(path):
    parts = path.split("/")
    node = RAY_DALIO_GAUGE_TREES[parts[0]]
    for p in parts[1:]:
        node = node[0][p]
    return node


def extend_alt(path, first_code, *extra):
    """Append fallback codes to the alternatives tuple (inside leaf `path`) that contains `first_code`."""
    comps = _node(path)[0]
    for i, comp in enumerate(comps):
        if isinstance(comp, tuple) and first_code in comp:
            comps[i] = comp + tuple(c for c in extra if c not in comp)
            return
    raise KeyError(f"{first_code} not found in {path}")


def rebuild_code_leaves():
    CODE_LEAVES.clear()
    for t, tree in RAY_DALIO_GAUGE_TREES.items():
        for path, comps, w in iter_leaves(tree, (t,)):
            for comp in comps:
                if not isinstance(comp, str):
                    for code_ in comp:
                        CODE_LEAVES[code_].append(path)


extend_alt("internal_order_cycle/wealth_values_gap", "SI.POV.GINI", "PIP.GINI")              # WDI first, PIP line-up estimate where WDI has nothing
extend_alt("internal_order_cycle/wealth_values_gap", "SI.DST.10TH.10", "PIP.DECILE10")
_node("internal_order_cycle/internal_conflict")[0].append(alt("ILO.STRIKES_LOCKOUTS*"))      # days lost to labor disputes (report #13 / #60)
rebuild_code_leaves()
print("tree patched; weights that do not sum to 1:", check_weights(RAY_DALIO_GAUGE_TREES))

In [ ]:
# 15. Master assembly
def assemble_master(frames):
    frames = [f for f in frames if f is not None and not f.empty]
    m = pd.concat(frames, ignore_index=True)
    m["is_projection"] = m["is_projection"].fillna(False).astype(bool) if "is_projection" in m.columns else False
    m["year"] = m["year"].astype(int)
    m["value"] = pd.to_numeric(m["value"], errors="coerce")
    m = m.dropna(subset=["value"])
    # same country/series/year from two providers: first frame in the list wins (list order = priority)
    return m.drop_duplicates(["country_code", "indicator_code", "year"], keep="first").reset_index(drop=True)


all_dfs = [
    df_wb_final, df_wb_other, df_wb_long, df_wb_edstats, 
    df_imf_final, df_cofer, df_eia_final, df_vdem_final,
    df_ndgain_final, df_owid_final, df_ilo_final, df_bis_final, 
    df_whr, df_unhcr, df_oecd, df_pwt, 
]
all_dfs = [
    df_wb_final, df_wb_other, df_wb_long, df_wb_edstats, 
    df_imf_final, df_cofer, df_eia_final, df_vdem_final,
    df_ndgain_final, df_owid_final, df_ilo_final, df_bis_final, 
    df_whr, df_unhcr, df_oecd, df_pwt,
    df_pip, df_arwu
]
df_master = assemble_master(all_dfs)
print(f"Updated df_master: {len(df_master)} rows, {df_master['indicator_code'].nunique()} series, {df_master['country_code'].nunique()} countries/aggregates.")

In [ ]:
# 15b. Derived features
def build_derived(df, countries, last_actual=2024, horizon=10):
    rows = []

    def wide(*codes):
        sub = df[df["indicator_code"].isin(codes)]
        if sub.empty:
            return pd.DataFrame()
        return sub.pivot_table(index="year", columns="country_code", values="value", aggfunc="first").sort_index()

    def emit(code, series_by_country, year, proj=False):
        for c, v in series_by_country.items():
            if pd.notna(v) and np.isfinite(v):
                rows.append({"country_code": c, "indicator_code": code, "year": int(year), "value": float(v),
                             "source": "DERIVED", "is_projection": proj})

    # (a) working-age population: yoy growth (history) + forward CAGR (report p.148) + dependency-ratio change (p.158)
    wa = wide("SP.POP.1564.TO", "PROJ.SP.POP.1564.TO")
    if not wa.empty:
        wa = wa.reindex(range(int(wa.index.min()), int(wa.index.max()) + 1))
        yoy = wa.pct_change(fill_method=None) * 100
        for y in yoy.index[(yoy.index <= last_actual)]:
            emit("DERIVED.WORKING_AGE_GROWTH_YOY", yoy.loc[y].dropna(), y)
        if last_actual in wa.index and last_actual + horizon in wa.index:
            emit("DERIVED.WORKING_AGE_GROWTH_FWD10", ((wa.loc[last_actual + horizon] / wa.loc[last_actual]) ** (1 / horizon) - 1) * 100, last_actual)
    dr = wide("SP.POP.DPND", "PROJ.SP.POP.DPND")
    if not dr.empty and last_actual in dr.index and last_actual + horizon in dr.index:
        emit("DERIVED.DEPENDENCY_RATIO_CHG_FWD10", dr.loc[last_actual + horizon] - dr.loc[last_actual], last_actual)

    # (b) trailing-10y volatility, capital productivity (needs the 1990+ history from cell 1c)
    def trailing(w, n=10):
        return w[w.index <= last_actual].tail(n) if not w.empty else w

    g, infl, inv = trailing(wide("NY.GDP.MKTP.KD.ZG")), trailing(wide("FP.CPI.TOTL.ZG")), trailing(wide("NE.GDI.FTOT.ZS"))
    for code, w in (("DERIVED.GDP_GROWTH_VOL10", g), ("DERIVED.INFLATION_VOL10", infl)):
        if not w.empty:
            emit(code, w.std()[w.notna().sum() >= 8], last_actual)
    if not g.empty and not inv.empty:
        emit("DERIVED.CAPITAL_PRODUCTIVITY", (g.mean() / inv.mean())[(g.notna().sum() >= 8) & (inv.notna().sum() >= 8)], last_actual)

    # (c) capital stock growth by perpetual inventory from constant-price GFCF (depreciation 5%, initial stock I0 / (g + delta), g = 5%)
    gfcf = wide("NE.GDI.FTOT.KD")
    if not gfcf.empty:
        delta, g0 = 0.05, 0.05
        for c in gfcf.columns:
            s = gfcf[c].dropna()
            s = s[s.index <= last_actual + 1]
            if len(s) < 20 or (s.index.max() - s.index.min() + 1) != len(s):
                continue
            k = [s.iloc[0] / (g0 + delta)]
            for inv_t in s.iloc[1:]:
                k.append(k[-1] * (1 - delta) + inv_t)
            kg = pd.Series(k, index=s.index).pct_change() * 100
            emit("DERIVED.CAPITAL_STOCK_GROWTH_PIM", {c: kg[kg.index <= last_actual].tail(5).mean()}, last_actual)

    # (d) debt growth relative to income growth: 5y change in debt/GDP. BIS total NFS credit first, then private credit + general government debt for the rest
    covered = set()
    priv, gov = wide("FS.AST.PRVT.GD.ZS"), wide("IMF.GGXWDG_NGDP")
    alt_ratio = (priv + gov) if not priv.empty and not gov.empty else pd.DataFrame()
    for w in (wide("BIS.TC.N.PCT_GDP"), alt_ratio):
        if w.empty:
            continue
        for c in w.columns:
            if c in covered:
                continue
            s = w[c].dropna()
            s = s[s.index <= last_actual + 1]
            if len(s) >= 6 and s.index.max() - 5 in s.index:
                emit("DERIVED.DEBT_GROWTH_MINUS_INCOME_GROWTH", {c: s.iloc[-1] - s.loc[s.index.max() - 5]}, s.index.max())
                covered.add(c)

    # (e) displacement share of population
    disp, pop = wide("UNHCR.DISPLACED_TOTAL"), wide("SP.POP.TOTL")
    if not disp.empty and not pop.empty:
        common = disp.index.intersection(pop.index)
        share = disp.loc[common] / pop.loc[common, [c for c in disp.columns if c in pop.columns]] * 100
        for y in share.index:
            emit("UNHCR.DISPLACED_PCT_POP", share.loc[y].dropna(), y)

    # (f) structural zeros: nuclear stockpile for non-nuclear states
    nuc = df[df["indicator_code"] == "FAS.NUCLEAR_STOCKPILE"]
    if not nuc.empty:
        have, yrs = set(nuc["country_code"]), sorted(nuc["year"].unique())
        for c in countries:
            if c not in have:
                for y in yrs:
                    rows.append({"country_code": c, "indicator_code": "FAS.NUCLEAR_STOCKPILE", "year": int(y),
                                 "value": 0.0, "source": "FAS-derived", "is_projection": False})
    return pd.DataFrame(rows)


df_derived = build_derived(df_master, target_countries)
print(df_derived.groupby("indicator_code").agg(rows=("value", "size"), countries=("country_code", "nunique")))
df_master = assemble_master([df_master, df_derived])      # re-assemble so derived series enter the health table

In [ ]:
# 16. Indicator health table
total_target_count = len(target_countries)
RECENT_FROM = 2023

if 'iso3' not in df_obs.columns:
    iso2_to_iso3 = target_codes.set_index("code")["iso3"].to_dict()
    df_obs['iso3'] = df_obs['code'].map(iso2_to_iso3)
obs_counts = df_obs['iso3'].value_counts()
country_weights = pd.Series(0, index=target_countries, dtype=float)
for c, cnt in obs_counts.items():
    if c in country_weights.index:
        country_weights[c] = cnt
country_weight_shares = country_weights / country_weights.sum()

records = []
proj_flag = df_master.groupby("indicator_code")["is_projection"].any()
for ind, grp in df_master.groupby("indicator_code"):
    grp = grp[~grp["is_projection"]]
    if grp.empty:
        continue
    rep = set(grp["country_code"].unique())
    recent = set(grp[grp["year"] >= RECENT_FROM]["country_code"].unique())
    std_val, distinct = grp["value"].std(), grp["value"].nunique()
    records.append({
        "gauge": resolve_gauge(ind),
        "indicator_code": ind,
        "source": grp["source"].iloc[0],
        "reporting_countries": len(rep & set(target_countries)),
        "country_cov_%": round(len(rep & set(target_countries)) / total_target_count * 100, 1),
        "weighted_cov_%": round(country_weight_shares[country_weight_shares.index.isin(rep)].sum() * 100, 1),
        f"recent_cov_% (>={RECENT_FROM})": round(len(recent & set(target_countries)) / total_target_count * 100, 1),
        f"weighted_recent_% (>={RECENT_FROM})": round(country_weight_shares[country_weight_shares.index.isin(recent)].sum() * 100, 1),
        "latest_year": grp["year"].max(),
        "degenerate": bool((std_val <= 0) and (distinct < 5)),
        "has_projection": bool(proj_flag[ind]),
    })

df_health = pd.DataFrame(records).sort_values(by=["gauge", "weighted_cov_%", f"weighted_recent_% (>={RECENT_FROM})"], ascending=[True, False, False])
display(df_health[['latest_year', 'weighted_cov_%', f'weighted_recent_% (>={RECENT_FROM})']].describe())

# leaf coverage on Dalio's 35 countries (+ Eurozone aggregate)
leaf_rep = leaf_coverage(RAY_DALIO_GAUGE_TREES, df_master, DALIO_35 + ["EMU"])
display(leaf_rep.sort_values("eff_weight", ascending=False))
display(leaf_rep.assign(gauge=leaf_rep.leaf.str.split("/").str[0]).groupby("gauge").apply(
    lambda x: (x.eff_weight * x.weight_share_available).sum() / x.eff_weight.sum()).rename("share_of_weight_with_data"))
print("weights that do not sum to 1:", check_weights(RAY_DALIO_GAUGE_TREES))

In [ ]:
# 17. Report coverage check (the report's own 64-item data-source list, pp.183-184)
def _by_name(pattern):
    return sorted(c for c, nm in wb_series_names.items() if re.search(pattern, nm, re.I))


REPORT_MAP = {
    1:  ("Access to essential health services (WB UHC index)",         ["SH.UHC.SRVS.CV.XD"]),
    3:  ("Agriculture, forestry & fishing value added",               ["NV.AGR.TOTL.ZS"]),
    8:  ("Child mortality rate",                                       ["SH.DYN.MORT"]),
    9:  ("Climate vulnerability & preparedness (+ subcomponents)",    ["NDGAIN.GAIN", "NDGAIN.VULNERABILITY", "NDGAIN.READINESS"]),
    10: ("CO2 emissions",                                              ["EN.GHG.CO2.MT.CE.AR5", "EN.GHG.CO2.PC.CE.AR5", "OWID.CO2_TOTAL_T", "OWID.CO2_PER_CAPITA_T"]),
    12: ("Construction permits (DB archive)",                          _by_name(r"construction permit|warehouse")),
    15: ("Education spending",                                         ["SE.XPD.TOTL.GD.ZS"]),
    16: ("Educational attainment (Barro-Lee / UIS)",                   ["BAR.SCHL.25UP", "SE.SEC.CUAT.UP.ZS", "SE.TER.CUAT.BA.ZS"]),
    17: ("Electricity production",                                     ["OWID.ELECTRICITY_GEN_TWH", "EIA.ELECTRICITY*"]),
    18: ("Energy production and consumption",                          ["EIA.ENERGY.PRODUCTION_QBTU", "EIA.ENERGY.SELF_SUFF", "OWID.PRIMARY_ENERGY_TWH"]),
    19: ("Enforcing contracts (DB archive)",                           _by_name(r"enforc")),
    20: ("GINI coefficient",                                           ["SI.POV.GINI"]),
    24: ("Gross expenditure on R&D",                                   ["GB.XPD.RSDV.GD.ZS"]),
    25: ("Aviation passengers & cargo",                                ["IS.AIR.PSGR", "IS.AIR.GOOD.MT.K1"]),
    27: ("Homicide rate",                                              ["VC.IHR.PSRC.P5"]),
    28: ("Hours worked per week",                                      ["ILO.HOURS_WEEKLY_MODELLED", "PWT.HOURS_PER_WORKER", "PWT.AVH"]),
    29: ("Human Capital Index",                                        ["HD.HCI.OVRL", "HD.HCI.EYRS", "HD.HCI.HLOS"]),
    30: ("Income distributions",                                       ["SI.DST.FRST.20", "SI.DST.05TH.20", "SI.DST.10TH.10"]),
    32: ("Internet users",                                             ["IT.NET.USER.ZS"]),
    33: ("Male labor force participation",                             ["SL.TLF.CACT.MA.ZS"]),
    35: ("Nuclear weapon stockpile",                                   ["FAS.NUCLEAR_STOCKPILE"]),
    39: ("Particulate matter pollution",                               ["EN.ATM.PM25.MC.M3"]),
    40: ("Patent applications",                                        ["IP.PAT.RESD", "IP.PAT.NRES"]),
    41: ("People displaced by conflict",                               ["UNHCR.DISPLACED_TOTAL"]),
    42: ("Safely managed drinking water",                              ["SH.H2O.SMDW.ZS"]),
    43: ("Prevalence of undernourishment",                             ["SN.ITK.DEFC.ZS"]),
    44: ("Protecting investors (DB archive)",                          _by_name(r"disclosure|legal rights|investor|minority")),
    46: ("Public sector corruption & theft (V-Dem)",                   ["VDEM.v2excrptps", "VDEM.v2exthftps"]),
    47: ("Rail lines and rail cargo",                                  ["IS.RRS.TOTL.KM", "IS.RRS.GOOD.MT.K6"]),
    48: ("Real GDP per worker",                                        ["PWT.RGDPO_PER_WORKER", "SL.GDP.PCAP.EM.KD"]),
    49: ("Real growth rate",                                           ["NY.GDP.MKTP.KD.ZG", "IMF.NGDP_RPCH"]),
    50: ("Renewable freshwater resources",                             ["ER.H2O.INTR.K3", "ER.H2O.INTR.PC"]),
    51: ("Rigorous & impartial government (V-Dem)",                    ["VDEM.v2clrspct"]),
    52: ("Royalty and license fees",                                   ["BX.GSR.ROYL.CD", "BM.GSR.ROYL.CD"]),
    55: ("Starting a business (DB archive)",                           _by_name(r"start a business|starting a business|business start|start-up")),
    56: ("Suicide mortality rate",                                     ["SH.STA.SUIC.P5"]),
    57: ("Tariff rate",                                                ["TM.TAX.MRCH.WM.AR.ZS"]),
    58: ("Top income shares (top 10% only; no top 1%)",                ["SI.DST.10TH.10"]),
    61: ("Transparent laws with predictable enforcement (V-Dem)",      ["VDEM.v2cltrnslw"]),
    63: ("# new businesses / people creating new businesses",          ["IC.BUS.NREG", "IC.BUS.NDNS.ZS"]),
    # not in the 64-item list but part of the report's gauges
    "H1": ("Life satisfaction (Cantril ladder)",                       ["WHR.CANTRIL_LADDER", "WHR.LIFE_LADDER"]),
    "H2": ("Social support / affect (Gallup via WHR)",                 ["WHR.SOCIAL_SUPPORT", "WHR.POSITIVE_AFFECT"]),
    "D1": ("Total credit to non-financial sector (BIS)",               ["BIS.TC.N.PCT_GDP", "BIS.TC.P.PCT_GDP", "BIS.TC.G.PCT_GDP"]),
    "D2": ("Central-bank policy rates (BIS)",                          ["BIS.CBPOL.POLICY_RATE_END"]),
    "R1": ("Reserve currency share (IMF COFER)",                       ["IMF.COFER_SHARE"]),
    "S1": ("Household saving rate (OECD)",                             ["OECD.HH_SAVINGS_RATE"]),
}
REPORT_MAP.update({
    3:  ("Agriculture, forestry & fishing value added", ["NV.AGR.TOTL.ZS", "NV.AGR.TOTL.CD"]),
    12: ("Construction permits (DB archive)",           ["IC.CNST.PRMT.DFRN*"]),
    13: ("Days lost to labor disputes (ILO strikes)",    ["ILO.STRIKES_LOCKOUTS*"]),
    19: ("Enforcing contracts (DB archive)",            ["ENF.CONT.COEN.DB*"]),
    20: ("GINI coefficient",                            ["SI.POV.GINI", "PIP.GINI"]),
    30: ("Income distributions",                        ["SI.DST.FRST.20", "SI.DST.05TH.20", "SI.DST.10TH.10", "PIP.DECILE1", "PIP.DECILE10"]),
    44: ("Protecting investors (DB archive)",           ["PROT.MINOR.INV.DFRN*"]),
    53: ("Share of universities in top 500 (ARWU)",     ["ARWU.TOP500_COUNT"]),
    55: ("Starting a business (DB archive)",            ["IC.REG.DFRN*"]),
    58: ("Top 10% income share (top 1% needs WID)",     ["SI.DST.10TH.10", "PIP.DECILE10"]),
    60: ("Total labor disputes (ILO strikes)",           ["ILO.STRIKES_LOCKOUTS*"]),
})

n_countries = df_master.groupby("indicator_code")["country_code"].nunique()
rows = []
for no, (concept, codes) in REPORT_MAP.items():
    codes = [h for pat in codes for h in _code_matches(pat, list(n_countries.index))] or list(codes)
    best = max([int(n_countries.get(c, 0)) for c in codes], default=0)
    rows.append({"report_#": no, "concept": concept, "codes": ", ".join(codes[:6]) + (" ..." if len(codes) > 6 else ""),
                 "max_countries_with_data": best, "status": "OK" if best else "EMPTY"})
df_report_cov = pd.DataFrame(rows)
display(df_report_cov)
implemented = {k for k in REPORT_MAP if isinstance(k, int)}
print(f"\nReport sources (of 64) with an implemented series: {len(implemented)}; with data in df_master: "
      f"{int((df_report_cov[df_report_cov['report_#'].apply(lambda x: isinstance(x, int))]['status'] == 'OK').sum())}")
print("Not implemented (report numbers):", sorted(set(range(1, 65)) - implemented))
print("\nFetch log:"); print_fetch_log()

In [ ]:
# 18. Vetting plot: how many indicators survive as the coverage thresholds tighten
import plotly.graph_objects as go

temp = df_health[df_health['latest_year'] >= RECENT_FROM].copy()
cov_cols = ['weighted_cov_%', f'weighted_recent_% (>={RECENT_FROM})']
z_scores, indicator_counts, gauge_counts, coverages, recent_coverages = [], [], [], [], []
for z in np.arange(0.01, 3.01, 0.01):
    for col in cov_cols:
        iqr = temp[col].quantile(0.75) - temp[col].quantile(0.25)
        temp[f'fail_{col}'] = temp[col] < (temp[col].median() - iqr * z)
    keep = temp[temp[[f'fail_{c}' for c in cov_cols]].sum(axis=1) == 0]       # only the fail_ flags (not has_projection / degenerate)
    z_scores.append(z)
    indicator_counts.append(len(keep))
    gauge_counts.append(keep['gauge'].nunique())
    coverages.append(keep['weighted_cov_%'].min() if len(keep) else np.nan)
    recent_coverages.append(keep[f'weighted_recent_% (>={RECENT_FROM})'].min() if len(keep) else np.nan)

for name, data in [("Indicator Counts", indicator_counts), ("Gauge Counts", gauge_counts),
                   ("Weighted Coverage (%)", coverages), ("Weighted Recent Coverage (%)", recent_coverages)]:
    fig = go.Figure(go.Scatter(x=z_scores, y=data, mode='lines'))
    fig.update_layout(title=name, xaxis_title="Z-Score Multiplier", yaxis_title=name, template="plotly_white")
    fig.show()